In [11]:
# ══════════════════════════════════════════════════════════════════════════════
#  06_WILLIE_HospitalPipeline.ipynb — CELL 1: Setup & Architecture
# ══════════════════════════════════════════════════════════════════════════════
#
#  PURPOSE: Define everything needed for hospital wound analysis inference.
#           Upload/take photo → classify + segment + detect wounds.
#
#  MODELS:
#    • WILLIE-MINI  (31.8M)  — fast edge deployment, 224×224
#    • WILLIE-BASE  (189M)   — best classifier, 378×378, 5-fold ensemble
#    • WILLIE-XL    (622M)   — multi-task (cls+seg+det), 378×378
#
#  STRATEGY:
#    Classification → BASE 5-fold TTA ensemble (90.60%) + XL ensemble
#    Segmentation  → XL (85.12% Dice)
#    Detection     → XL seg mask → connected components → bboxes
#
# ══════════════════════════════════════════════════════════════════════════════

import os, sys, math, time, warnings, json, glob
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from collections import OrderedDict
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")

print("=" * 80)
print("  06_WILLIE_HospitalPipeline — Cell 1: Setup & Architecture")
print(f"  Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION A: DEVICE & PATHS
# ══════════════════════════════════════════════════════════════════════════════

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n⚡ Device: {DEVICE}")
if DEVICE.type == "cuda":
    gpu_name = torch.cuda.get_device_name()
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   GPU: {gpu_name} ({gpu_mem:.1f} GB)")

# ── Project paths ──
PROJECT_ROOT = Path(".")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "woundshot_v2"
MANIFESTS_DIR = ARTIFACTS_DIR / "manifests"
SAM2_MASKS_DIR = ARTIFACTS_DIR / "sam2_masks"
FIGURES_DIR = ARTIFACTS_DIR / "figures"
CHECKPOINT_DIR = ARTIFACTS_DIR / "checkpoints" / "transformer"
XL_CHECKPOINT_DIR = ARTIFACTS_DIR / "checkpoints" / "xl"
PIPELINE_DIR = ARTIFACTS_DIR / "hospital_pipeline"

# Create output dirs
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ──
NUM_CLASSES = 5
CLASS_NAMES = ["diabetic", "pressure", "surgical", "venous", "no_wound"]
IDX_TO_CLASS = {i: c for i, c in enumerate(CLASS_NAMES)}
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

# ImageNet normalization
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

print(f"\n📂 Paths:")
print(f"   Project:     {PROJECT_ROOT}")
print(f"   Artifacts:   {ARTIFACTS_DIR}")
print(f"   Checkpoints: {CHECKPOINT_DIR}")
print(f"   XL Ckpts:    {XL_CHECKPOINT_DIR}")
print(f"   Pipeline:    {PIPELINE_DIR}")


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION B: CHECKPOINT DISCOVERY
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n🔍 CHECKPOINT DISCOVERY")
print("-" * 80)

discovered_checkpoints = {
    "mini": [],
    "base": {"cell07": [], "cell07b": [], "cell07c": []},
    "xl": [],
}

# ── BASE checkpoints (5-fold × 3 versions) ──
if CHECKPOINT_DIR.exists():
    for variant in ["cell07", "cell07b", "cell07c"]:
        for fold in range(1, 6):
            path = CHECKPOINT_DIR / f"{variant}_fold{fold}_best.pt"
            if path.exists():
                size_mb = path.stat().st_size / (1024**2)
                discovered_checkpoints["base"][variant].append({
                    "fold": fold, "path": str(path), "size_mb": size_mb
                })

    # Print BASE summary
    for variant, folds in discovered_checkpoints["base"].items():
        if folds:
            print(f"  ✅ BASE {variant}: {len(folds)} fold(s) found")
            for f in folds:
                print(f"       Fold {f['fold']}: {f['size_mb']:.1f} MB")
        else:
            print(f"  ❌ BASE {variant}: not found")
else:
    print(f"  ❌ Checkpoint dir not found: {CHECKPOINT_DIR}")

# ── XL checkpoints ──
xl_search_paths = [
    XL_CHECKPOINT_DIR,
    CHECKPOINT_DIR,
    ARTIFACTS_DIR / "woundshot_runs",
    ARTIFACTS_DIR / "checkpoints",
]
# Broad pattern: any file starting with "xl_"
xl_patterns = ["xl_*.pt"]

seen_paths = set()
for search_dir in xl_search_paths:
    if search_dir.exists():
        for pat in xl_patterns:
            for match in search_dir.glob(pat):
                if str(match) not in seen_paths:
                    seen_paths.add(str(match))
                    size_mb = match.stat().st_size / (1024**2)
                    discovered_checkpoints["xl"].append({
                        "name": match.name, "path": str(match), "size_mb": size_mb
                    })

# Also do a recursive search in artifacts if still nothing
if not discovered_checkpoints["xl"]:
    for match in ARTIFACTS_DIR.rglob("xl_*.pt"):
        if str(match) not in seen_paths:
            seen_paths.add(str(match))
            size_mb = match.stat().st_size / (1024**2)
            discovered_checkpoints["xl"].append({
                "name": match.name, "path": str(match), "size_mb": size_mb
            })

# Categorize XL checkpoints
xl_model_ckpts = [c for c in discovered_checkpoints["xl"]
                  if c["name"] not in ("xl_test_probs.pt", "xl_training_history.pt")]
xl_probs_ckpts = [c for c in discovered_checkpoints["xl"]
                  if c["name"] == "xl_test_probs.pt"]

if discovered_checkpoints["xl"]:
    print(f"\n  ✅ XL: {len(discovered_checkpoints['xl'])} file(s) found")
    for ckpt in discovered_checkpoints["xl"]:
        tag = ""
        if "test_probs" in ckpt["name"]:
            tag = " (pre-computed probabilities)"
        elif "history" in ckpt["name"]:
            tag = " (training history)"
        print(f"       {ckpt['name']}: {ckpt['size_mb']:.1f} MB{tag}")

    if xl_model_ckpts:
        print(f"  📊 XL model checkpoints: {len(xl_model_ckpts)}")
    if xl_probs_ckpts:
        print(f"  📊 XL pre-computed probs: {len(xl_probs_ckpts)} (can use for ensemble)")
else:
    print(f"\n  ⚠️  XL: No checkpoints found (will skip XL inference)")

# ── Summary ──
best_base_variant = None
best_base_count = 0
for variant, folds in discovered_checkpoints["base"].items():
    if len(folds) > best_base_count:
        best_base_count = len(folds)
        best_base_variant = variant

print(f"\n  📊 Best BASE variant: {best_base_variant} ({best_base_count} folds)")
HAS_XL_MODEL = len(xl_model_ckpts) > 0
HAS_XL_PROBS = len(xl_probs_ckpts) > 0
HAS_XL = HAS_XL_MODEL or HAS_XL_PROBS
print(f"  📊 XL model weights: {'Yes' if HAS_XL_MODEL else 'No'}")
print(f"  📊 XL pre-computed probs: {'Yes' if HAS_XL_PROBS else 'No'}")


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION C: MODEL ARCHITECTURES
# ══════════════════════════════════════════════════════════════════════════════

# ── C.1: Shared Components ──

print(f"\n\n🏗️  MODEL ARCHITECTURES")
print("-" * 80)


# ──────────────────────────────────────────────────────────────────────────────
#  C.1: WILLIE-BASE v2 Architecture
# ──────────────────────────────────────────────────────────────────────────────

@dataclass
class BaseV2Config:
    """WILLIE-BASE v2 configuration."""
    backbones: List = field(default_factory=lambda: [
        {"name": "vit_base_patch14_dinov2", "out_dim": 768, "model_type": "vit"},
        {"name": "convnext_base", "out_dim": 1024, "model_type": "cnn"},
    ])
    img_size: int = 378
    fusion_dim: int = 384
    num_fusion_layers: int = 3
    num_heads: int = 12
    num_freq_bands: int = 4
    dropout: float = 0.20
    use_wound_gating: bool = True
    num_classes: int = 5
    class_names: List[str] = field(default_factory=lambda: [
        "diabetic", "pressure", "surgical", "venous", "no_wound"
    ])


class DualBackboneEncoder(nn.Module):
    """Dual backbone: DINOv2-ViT-B + ConvNeXt-Base → projected token sequences."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.backbones = nn.ModuleList()
        self.projections = nn.ModuleList()

        for bc in cfg.backbones:
            if isinstance(bc, dict):
                name, out_dim, mtype = bc["name"], bc["out_dim"], bc["model_type"]
            else:
                name, out_dim, mtype = bc.name, bc.out_dim, bc.model_type

            if mtype == "vit":
                bb = timm.create_model(name, pretrained=True,
                                       num_classes=0, global_pool='',
                                       dynamic_img_size=True)
            else:
                bb = timm.create_model(name, pretrained=True,
                                       num_classes=0, global_pool='')
            self.backbones.append(bb)
            self.projections.append(nn.Sequential(
                nn.Linear(out_dim, cfg.fusion_dim),
                nn.LayerNorm(cfg.fusion_dim),
                nn.GELU(),
                nn.Dropout(cfg.dropout * 0.5),
            ))

    def forward(self, x):
        features = []
        for i, (bb, proj) in enumerate(zip(self.backbones, self.projections)):
            bc = self.cfg.backbones[i]
            mtype = bc["model_type"] if isinstance(bc, dict) else bc.model_type
            feat = bb(x)
            if mtype == "cnn" and feat.dim() == 4:
                B, C, H, W = feat.shape
                feat = feat.flatten(2).transpose(1, 2)
            elif feat.dim() == 3 and mtype == "vit":
                if feat.shape[1] > 50:
                    feat = feat[:, 1:, :]  # remove CLS token
            features.append(proj(feat))
        return features


class FreqDecomposedCrossAttention(nn.Module):
    """WA-FDCA: Wound-Aware Frequency-Decomposed Cross-Attention."""

    def __init__(self, dim, num_heads=8, num_freq_bands=4,
                 dropout=0.1, use_wound_gating=True):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.num_freq_bands = num_freq_bands
        self.use_wound_gating = use_wound_gating

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

        self.freq_filters = nn.Parameter(
            torch.randn(num_freq_bands, 1, num_heads, 1, 1) * 0.02)
        self.freq_combine = nn.Linear(num_freq_bands * dim, dim)

        if use_wound_gating:
            self.wound_gate = nn.Sequential(
                nn.Linear(dim, dim // 4), nn.GELU(),
                nn.Linear(dim // 4, num_heads), nn.Sigmoid(),
            )

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, context, wound_prior=None):
        B, N, C = x.shape
        _, M, _ = context.shape

        x_norm = self.norm1(x)
        ctx_norm = self.norm2(context)

        q = self.q_proj(x_norm).reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(ctx_norm).reshape(B, M, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(ctx_norm).reshape(B, M, self.num_heads, self.head_dim).transpose(1, 2)

        scale = self.head_dim ** -0.5
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale

        if self.use_wound_gating and wound_prior is not None:
            gate = self.wound_gate(wound_prior.mean(dim=1))
            attn = attn * gate.unsqueeze(-1).unsqueeze(-1)

        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).reshape(B, N, C)
        out = self.out_proj(out)
        return x + out


class WILLIEBaseV2(nn.Module):
    """WILLIE-BASE v2: Dual backbone + WA-FDCA fusion + classification."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.encoder = DualBackboneEncoder(cfg)

        self.cross_attn_layers = nn.ModuleList([
            FreqDecomposedCrossAttention(
                dim=cfg.fusion_dim, num_heads=cfg.num_heads,
                num_freq_bands=cfg.num_freq_bands,
                dropout=cfg.dropout, use_wound_gating=cfg.use_wound_gating,
            )
            for _ in range(cfg.num_fusion_layers)
        ])

        self.cls_norm = nn.LayerNorm(cfg.fusion_dim)
        self.cls_head = nn.Sequential(
            nn.Linear(cfg.fusion_dim, cfg.fusion_dim),
            nn.GELU(), nn.Dropout(cfg.dropout),
            nn.Linear(cfg.fusion_dim, cfg.fusion_dim // 2),
            nn.GELU(), nn.Dropout(cfg.dropout * 0.5),
            nn.Linear(cfg.fusion_dim // 2, cfg.num_classes),
        )

    def forward(self, x, task="classification"):
        features = self.encoder(x)
        if len(features) >= 2:
            fused = features[0]
            for layer in self.cross_attn_layers:
                fused = layer(fused, features[1], wound_prior=features[0])
        else:
            fused = features[0]
        pooled = self.cls_norm(fused.mean(dim=1))
        logits = self.cls_head(pooled)
        return {"cls_logits": logits}


print("  ✅ WILLIEBaseV2 defined (DualBackbone + WA-FDCA + cls)")


# ──────────────────────────────────────────────────────────────────────────────
#  C.2: WILLIE-XL Architecture
# ──────────────────────────────────────────────────────────────────────────────

@dataclass
class XLConfig:
    """WILLIE_XL configuration."""
    backbone_1: str = "vit_large_patch14_dinov2.lvd142m"
    backbone_2: str = "convnext_large.fb_in22k_ft_in1k"
    backbone_3: str = "sam2"
    backbone_3_fallback: str = "swin_base_patch4_window7_224.ms_in22k_ft_in1k"

    bb1_dim: int = 1024
    bb2_dim: int = 1536
    bb3_dim: int = 768

    img_size: int = 378
    fusion_dim: int = 512
    num_fdca_layers: int = 3
    num_freq_bands: int = 4
    num_heads: int = 16
    dropout: float = 0.25
    use_gradient_ckpt: bool = True

    num_wacsa_layers: int = 2
    wacsa_scales: int = 3

    num_experts: int = 6
    top_k_experts: int = 2
    moe_hidden: int = 512

    seg_channels: List[int] = field(default_factory=lambda: [256, 128, 64])
    film_dim: int = 256

    num_det_queries: int = 50
    det_hidden: int = 256
    det_num_layers: int = 3

    embed_dim: int = 128
    wbrn_channels: int = 64
    wbrn_dilations: List[int] = field(default_factory=lambda: [1, 2, 4, 8])

    num_classes: int = 5
    class_names: List[str] = field(default_factory=lambda: [
        "diabetic", "pressure", "surgical", "venous", "no_wound"
    ])

    mc_dropout_passes: int = 10
    uncertainty_threshold: float = 0.3

    batch_size: int = 4
    accumulation_steps: int = 8
    base_lr: float = 1e-4
    num_epochs: int = 150
    backbone_lr_scale: float = 0.05


# XL sub-components — only loaded if XL checkpoints exist

class TripleBackboneEncoder(nn.Module):
    """Triple backbone: DINOv2-ViT-L + ConvNeXt-L + SAM2-Hiera/Swin."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        # BB1: DINOv2 ViT-L
        self.bb1 = timm.create_model(
            cfg.backbone_1, pretrained=True, num_classes=0, global_pool="")
        with torch.no_grad():
            dummy = torch.randn(1, 3, cfg.img_size, cfg.img_size)
            out1 = self.bb1(dummy)
            cfg.bb1_dim = out1.shape[-1] if out1.dim() == 3 else out1.shape[1]
        print(f"     BB1: {cfg.backbone_1.split('.')[0]} → {cfg.bb1_dim}d")

        # BB2: ConvNeXt-L (features_only for multi-scale)
        self.bb2 = timm.create_model(
            cfg.backbone_2, pretrained=True, features_only=True)
        with torch.no_grad():
            out2 = self.bb2(dummy)
            cfg.bb2_dim = out2[-1].shape[1]
            self.bb2_scales = [(o.shape[1], o.shape[2], o.shape[3]) for o in out2]
        print(f"     BB2: {cfg.backbone_2.split('.')[0]} → {cfg.bb2_dim}d "
              f"(scales: {self.bb2_scales})")

        # BB3: Try SAM2-Hiera, fallback to other
        bb3_loaded = False
        bb3_candidates = [
            "hiera_base_plus_224.mae_in1k_ft_in1k",
            "hiera_base_224.mae_in1k_ft_in1k",
            cfg.backbone_3_fallback,
        ]
        for bb3_name in bb3_candidates:
            try:
                self.bb3 = timm.create_model(
                    bb3_name, pretrained=True, num_classes=0, global_pool="")
                with torch.no_grad():
                    out3 = self.bb3(dummy)
                    if out3.dim() == 3:
                        cfg.bb3_dim = out3.shape[-1]
                    elif out3.dim() == 4:
                        cfg.bb3_dim = out3.shape[1]
                    else:
                        cfg.bb3_dim = out3.shape[-1]
                cfg.backbone_3 = bb3_name
                bb3_loaded = True
                print(f"     BB3: {bb3_name} → {cfg.bb3_dim}d")
                break
            except Exception as e:
                print(f"     ⚠️  {bb3_name} failed ({type(e).__name__}), trying next...")
                continue

        if not bb3_loaded:
            # Last resort: use a lightweight CNN
            self.bb3 = timm.create_model(
                "efficientnet_b3", pretrained=True, num_classes=0, global_pool="avg")
            with torch.no_grad():
                out3 = self.bb3(dummy)
                cfg.bb3_dim = out3.shape[-1] if out3.dim() == 2 else out3.shape[1]
            cfg.backbone_3 = "efficientnet_b3"
            print(f"     BB3 (fallback): efficientnet_b3 → {cfg.bb3_dim}d")

        del dummy

        # Projections to fusion_dim
        self.proj1 = nn.Sequential(
            nn.Linear(cfg.bb1_dim, cfg.fusion_dim),
            nn.LayerNorm(cfg.fusion_dim), nn.GELU(),
            nn.Dropout(cfg.dropout * 0.3))
        self.proj2 = nn.Sequential(
            nn.Linear(cfg.bb2_dim, cfg.fusion_dim),
            nn.LayerNorm(cfg.fusion_dim), nn.GELU(),
            nn.Dropout(cfg.dropout * 0.3))
        self.proj3 = nn.Sequential(
            nn.Linear(cfg.bb3_dim, cfg.fusion_dim),
            nn.LayerNorm(cfg.fusion_dim), nn.GELU(),
            nn.Dropout(cfg.dropout * 0.3))

    def _to_tokens(self, feat, proj):
        """Convert any feature format to [B, N, D] tokens."""
        if feat.dim() == 4:  # [B, C, H, W] → [B, H*W, C]
            B, C, H, W = feat.shape
            feat = feat.flatten(2).transpose(1, 2)
        elif feat.dim() == 3:  # [B, N, D] — might have CLS token
            if feat.shape[1] > 50:
                feat = feat[:, 1:, :]  # drop CLS
        elif feat.dim() == 2:  # [B, D] → [B, 1, D]
            feat = feat.unsqueeze(1)
        return proj(feat)

    def forward(self, x):
        f1 = self.bb1(x)
        f2_scales = self.bb2(x)
        f2 = f2_scales[-1]  # Use final scale
        f3 = self.bb3(x)

        t1 = self._to_tokens(f1, self.proj1)
        t2 = self._to_tokens(f2, self.proj2)
        t3 = self._to_tokens(f3, self.proj3)

        return [t1, t2, t3], f2_scales


class F2DCA_Layer(nn.Module):
    """F²DCA: Feature-to-Feature Dense Cross-Attention with frequency decomposition."""

    def __init__(self, dim, num_heads=16, num_freq_bands=4, dropout=0.25):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

        self.freq_filters = nn.Parameter(
            torch.randn(num_freq_bands, 1, num_heads, 1, 1) * 0.02)
        self.freq_combine = nn.Linear(num_freq_bands * dim, dim)

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim * 4, dim), nn.Dropout(dropout))
        self.norm3 = nn.LayerNorm(dim)

    def forward(self, x, context):
        B, N, C = x.shape
        _, M, _ = context.shape

        q = self.q_proj(self.norm1(x)).reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(self.norm2(context)).reshape(B, M, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(self.norm2(context)).reshape(B, M, self.num_heads, self.head_dim).transpose(1, 2)

        scale = self.head_dim ** -0.5
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v).transpose(1, 2).reshape(B, N, C)
        out = self.out_proj(out)

        x = x + out
        x = x + self.ffn(self.norm3(x))
        return x


class WA_CSA_Layer(nn.Module):
    """Wound-Aware Cross-Scale Attention."""

    def __init__(self, dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.wound_gate = nn.Sequential(
            nn.Linear(dim, dim // 4), nn.GELU(),
            nn.Linear(dim // 4, 1), nn.Sigmoid())
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.alpha = nn.Parameter(torch.zeros(1))

    def forward(self, fine, coarse):
        gate = self.wound_gate(fine)
        gated_fine = fine * gate
        out, _ = self.attn(self.norm1(gated_fine), self.norm2(coarse), coarse)
        return fine + torch.tanh(self.alpha) * out


class MoEClassifier(nn.Module):
    """Mixture-of-Experts wound classifier."""

    def __init__(self, cfg):
        super().__init__()
        self.num_experts = cfg.num_experts
        self.top_k = cfg.top_k_experts

        self.router = nn.Linear(cfg.fusion_dim, cfg.num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(cfg.fusion_dim, cfg.moe_hidden), nn.GELU(),
                nn.Dropout(cfg.dropout),
                nn.Linear(cfg.moe_hidden, cfg.num_classes))
            for _ in range(cfg.num_experts)
        ])

    def forward(self, x):
        router_logits = self.router(x)
        topk_vals, topk_idx = torch.topk(router_logits, self.top_k, dim=-1)
        topk_weights = F.softmax(topk_vals, dim=-1)

        B = x.shape[0]
        all_expert_out = torch.stack([e(x) for e in self.experts], dim=1)
        topk_idx_exp = topk_idx.unsqueeze(-1).expand(-1, -1, all_expert_out.shape[-1])
        selected = torch.gather(all_expert_out, 1, topk_idx_exp)
        logits = (selected * topk_weights.unsqueeze(-1)).sum(dim=1)
        return logits


class FiLMConditioner(nn.Module):
    """FiLM: Feature-wise Linear Modulation for WTCS."""

    def __init__(self, cond_dim, feat_dim):
        super().__init__()
        self.gamma_proj = nn.Linear(cond_dim, feat_dim)
        self.beta_proj = nn.Linear(cond_dim, feat_dim)

    def forward(self, feat, cond):
        gamma = self.gamma_proj(cond)
        beta = self.beta_proj(cond)
        if feat.dim() == 4:
            gamma = gamma.unsqueeze(-1).unsqueeze(-1)
            beta = beta.unsqueeze(-1).unsqueeze(-1)
        return feat * (1 + gamma) + beta


class WTCSDecoder(nn.Module):
    """Wound-Type Conditioned Segmentation decoder with FiLM."""

    def __init__(self, cfg, ms_channels=None):
        super().__init__()
        D = cfg.fusion_dim
        self.token_to_spatial = nn.Linear(D, 128)
        self.film = FiLMConditioner(cfg.num_classes, 128)

        self.upsample = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.GELU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1),
        )

    def forward(self, fused_tokens, wound_logits, target_size=None):
        B, N, D = fused_tokens.shape
        h = int(math.sqrt(N))
        seg_feat = self.token_to_spatial(fused_tokens)
        seg_feat = self.film(seg_feat, F.softmax(wound_logits, dim=-1))
        seg_2d = seg_feat.reshape(B, h, h, 128).permute(0, 3, 1, 2)
        seg_2d = F.interpolate(seg_2d, size=(28, 28), mode='bilinear', align_corners=False)
        mask = self.upsample(seg_2d)
        if target_size is not None:
            mask = F.interpolate(mask, size=target_size, mode='bilinear', align_corners=False)
        return mask


class WBRN(nn.Module):
    """Wound Boundary Refinement Network."""

    def __init__(self, cfg):
        super().__init__()
        ch = cfg.wbrn_channels
        self.dilated_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(1, ch, 3, padding=d, dilation=d),
                nn.BatchNorm2d(ch), nn.GELU())
            for d in cfg.wbrn_dilations
        ])
        self.fuse = nn.Sequential(
            nn.Conv2d(ch * len(cfg.wbrn_dilations), ch, 1),
            nn.BatchNorm2d(ch), nn.GELU(),
            nn.Conv2d(ch, 1, 1))

    def forward(self, coarse_mask):
        feats = [conv(coarse_mask) for conv in self.dilated_convs]
        fused = torch.cat(feats, dim=1)
        refined = self.fuse(fused)
        return coarse_mask + refined


class DetectionDecoder(nn.Module):
    """DETR-style detection decoder."""

    def __init__(self, cfg):
        super().__init__()
        D = cfg.fusion_dim
        self.queries = nn.Parameter(torch.randn(1, cfg.num_det_queries, D) * 0.02)
        self.cross_attn = nn.MultiheadAttention(D, cfg.num_heads, dropout=cfg.dropout,
                                                 batch_first=True)
        self.box_head = nn.Sequential(
            nn.Linear(D, D), nn.GELU(), nn.Linear(D, 4), nn.Sigmoid())
        self.cls_head = nn.Sequential(
            nn.Linear(D, D // 2), nn.GELU(), nn.Linear(D // 2, 2))

    def forward(self, fused_tokens):
        B = fused_tokens.shape[0]
        queries = self.queries.expand(B, -1, -1)
        det_feat, _ = self.cross_attn(queries, fused_tokens, fused_tokens)
        boxes = self.box_head(det_feat)
        cls = self.cls_head(det_feat)
        return boxes, cls


class ContrastiveHead(nn.Module):
    """Contrastive embedding head for wound retrieval."""

    def __init__(self, cfg):
        super().__init__()
        self.projector = nn.Sequential(
            nn.Linear(cfg.fusion_dim, cfg.fusion_dim), nn.GELU(),
            nn.Linear(cfg.fusion_dim, cfg.embed_dim))

    def forward(self, pooled):
        return F.normalize(self.projector(pooled), p=2, dim=-1)


class WoundAttentionMaps(nn.Module):
    """WAM: Wound Attention Maps for clinical explainability."""

    def __init__(self, cfg):
        super().__init__()
        self.map_head = nn.Sequential(
            nn.Linear(cfg.fusion_dim, cfg.fusion_dim // 2), nn.GELU(),
            nn.Linear(cfg.fusion_dim // 2, 3), nn.Softmax(dim=-1))

    def forward(self, fused_tokens, spatial_shape):
        attn = self.map_head(fused_tokens)
        H, W = spatial_shape
        side = int(math.sqrt(attn.shape[1]))
        attn = attn.transpose(1, 2).view(-1, 3, side, side)
        attn = F.interpolate(attn, size=(H, W), mode="bilinear", align_corners=False)
        return attn


class WILLIEXL(nn.Module):
    """
    WILLIE_XL: Multi-Task Wound Analysis Transformer.
    8 Novel Components: Triple Backbone, F²DCA, WA-CSA, MoE,
    WTCS+FiLM, Contrastive, Uncertainty (MC Dropout), WBRN, WAM.
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        # 1. Triple Backbone
        self.encoder = TripleBackboneEncoder(cfg)

        # 2. F²DCA Fusion
        self.fdca_layers = nn.ModuleList([
            F2DCA_Layer(cfg.fusion_dim, cfg.num_heads, cfg.num_freq_bands, cfg.dropout)
            for _ in range(cfg.num_fdca_layers)
        ])

        # 3. WA-CSA
        self.wacsa_layers = nn.ModuleList([
            WA_CSA_Layer(cfg.fusion_dim, num_heads=8, dropout=cfg.dropout)
            for _ in range(cfg.num_wacsa_layers)
        ])
        self.coarse_pool = nn.AdaptiveAvgPool1d(64)

        # Pooling
        self.cls_pool = nn.Sequential(
            nn.LayerNorm(cfg.fusion_dim), nn.AdaptiveAvgPool1d(1), nn.Flatten())

        # 4. MoE Classification
        self.moe_classifier = MoEClassifier(cfg)

        # 5. WTCS Segmentation
        self.wtcs_decoder = WTCSDecoder(cfg)

        # 8. WBRN
        self.wbrn = WBRN(cfg)

        # Detection
        self.det_decoder = DetectionDecoder(cfg)

        # 6. Contrastive
        self.contrastive_head = ContrastiveHead(cfg)

        # WAM
        self.wam = WoundAttentionMaps(cfg)

    def _align_tokens(self, token_list):
        """Align all token sequences to same length via interpolation."""
        min_n = min(t.shape[1] for t in token_list)
        target_h = int(math.sqrt(min_n))
        target_n = target_h * target_h

        aligned = []
        for t in token_list:
            if t.shape[1] != target_n:
                B, N, D = t.shape
                h = int(math.sqrt(N))
                if h * h > N:
                    h -= 1
                t_trim = t[:, :h*h, :]
                t_2d = t_trim.reshape(B, h, h, D).permute(0, 3, 1, 2)
                t_2d = F.interpolate(t_2d, size=(target_h, target_h),
                                     mode='bilinear', align_corners=False)
                t = t_2d.permute(0, 2, 3, 1).reshape(B, target_n, D)
            aligned.append(t)
        return aligned

    def forward(self, x, task="all"):
        input_size = x.shape[-2:]  # (H, W)

        # 1. Triple backbone
        token_lists, cnn_scales = self.encoder(x)
        tokens = self._align_tokens(token_lists)

        # 2. F²DCA: pairwise cross-attention between backbones
        fused = tokens[0]
        for fdca in self.fdca_layers:
            for i in range(1, len(tokens)):
                fused = fdca(fused, tokens[i])

        # 3. WA-CSA: cross-scale refinement
        for wacsa in self.wacsa_layers:
            coarse = self.coarse_pool(fused.transpose(1, 2)).transpose(1, 2)
            fused = wacsa(fused, coarse)

        output = {}

        # 4. Classification (MoE)
        if task in ["classification", "all"]:
            pooled = self.cls_pool(fused.transpose(1, 2))
            cls_logits = self.moe_classifier(pooled)
            output["cls_logits"] = cls_logits

            # Refined classification (after seg feedback)
            output["cls_refined"] = cls_logits  # Updated below if seg runs

        # 5. Segmentation (WTCS + WBRN)
        if task in ["segmentation", "all"]:
            coarse_mask = self.wtcs_decoder(fused, output.get("cls_logits", torch.zeros(x.shape[0], self.cfg.num_classes, device=x.device)),
                                            target_size=input_size)
            refined_mask = self.wbrn(coarse_mask)
            output["seg_mask"] = refined_mask

        # Detection
        if task in ["detection", "all"]:
            det_boxes, det_cls = self.det_decoder(fused)
            output["det_boxes"] = det_boxes
            output["det_classes"] = det_cls

        # 6. Contrastive embeddings
        if task in ["all"]:
            pooled = self.cls_pool(fused.transpose(1, 2))
            output["embeddings"] = self.contrastive_head(pooled)

        # WAM
        if task in ["all", "explainability"]:
            output["wam"] = self.wam(fused, input_size)

        return output


print("  ✅ WILLIEXL defined (Triple Backbone + F²DCA + WA-CSA + MoE + WTCS + WBRN)")


# ──────────────────────────────────────────────────────────────────────────────
#  C.3: WILLIE-MINI Architecture (from notebook 04 v3)
# ──────────────────────────────────────────────────────────────────────────────

class DINOv2MultiScale(nn.Module):
    """Shared DINOv2 backbone with multi-scale feature extraction via hooks."""

    def __init__(self, model_name="dinov2_vits14", extract_layers=(2, 5, 8, 11)):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=True,
                                       num_classes=0, global_pool='',
                                       dynamic_img_size=True)
        self.extract_layers = extract_layers

        with torch.no_grad():
            dummy = torch.randn(1, 3, 224, 224)
            out = self.model(dummy)
            self.embed_dim = out.shape[-1]
        del dummy

        self._features = {}
        for idx in extract_layers:
            self.model.blocks[idx].register_forward_hook(
                self._make_hook(idx))

    def _make_hook(self, idx):
        def hook(module, input, output):
            self._features[idx] = output
        return hook

    def forward(self, x):
        self._features = {}
        _ = self.model(x)
        B = x.shape[0]
        h = int(math.sqrt(list(self._features.values())[0].shape[1] - 1))

        multi_scale = []
        for idx in self.extract_layers:
            feat = self._features[idx][:, 1:, :]  # drop CLS
            feat_2d = feat.reshape(B, h, h, -1).permute(0, 3, 1, 2)
            multi_scale.append(feat_2d)
        return multi_scale


class FeaturePyramidNeck(nn.Module):
    """FPN-style neck for multi-scale features."""

    def __init__(self, in_dim, neck_dim, num_levels=4):
        super().__init__()
        self.lateral_convs = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_dim, neck_dim, 1), nn.BatchNorm2d(neck_dim), nn.GELU())
            for _ in range(num_levels)
        ])
        self.output_convs = nn.ModuleList([
            nn.Sequential(nn.Conv2d(neck_dim, neck_dim, 3, padding=1),
                          nn.BatchNorm2d(neck_dim), nn.GELU())
            for _ in range(num_levels)
        ])

    def forward(self, features):
        laterals = [conv(f) for conv, f in zip(self.lateral_convs, features)]
        for i in range(len(laterals) - 2, -1, -1):
            up = F.interpolate(laterals[i + 1], size=laterals[i].shape[-2:],
                               mode='bilinear', align_corners=False)
            laterals[i] = laterals[i] + up
        return [conv(lat) for conv, lat in zip(self.output_convs, laterals)]


class WoundAwareCrossScaleAttention(nn.Module):
    """WA-CSA: Bidirectional cross-scale attention with wound gating."""

    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.attn_fine2coarse = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.attn_coarse2fine = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.wound_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(dim, dim // 4), nn.GELU(),
            nn.Linear(dim // 4, 1), nn.Sigmoid())
        self.alpha = nn.Parameter(torch.zeros(1))
        self.norm = nn.LayerNorm(dim)

    def forward(self, fine_feat, coarse_feat):
        B, C, Hf, Wf = fine_feat.shape
        _, _, Hc, Wc = coarse_feat.shape

        fine_tok = fine_feat.flatten(2).transpose(1, 2)
        coarse_tok = coarse_feat.flatten(2).transpose(1, 2)
        coarse_up_tok = F.interpolate(
            coarse_feat, size=(Hf, Wf), mode='bilinear', align_corners=False
        ).flatten(2).transpose(1, 2)

        gate = self.wound_gate(fine_feat)
        f2c, _ = self.attn_fine2coarse(fine_tok, coarse_up_tok, coarse_up_tok)
        c2f, _ = self.attn_coarse2fine(coarse_up_tok, fine_tok, fine_tok)

        refined = fine_tok + torch.tanh(self.alpha) * gate.unsqueeze(-1) * (f2c + c2f)
        refined = self.norm(refined)
        return refined.transpose(1, 2).reshape(B, C, Hf, Wf)


class WA_CSA_Stack(nn.Module):
    """Stack of WA-CSA layers processing adjacent pyramid levels."""

    def __init__(self, dim, num_heads=4, num_layers=2):
        super().__init__()
        self.layers = nn.ModuleList([
            WoundAwareCrossScaleAttention(dim, num_heads)
            for _ in range(num_layers)
        ])

    def forward(self, pyramid):
        for layer in self.layers:
            refined = []
            for i in range(len(pyramid)):
                if i + 1 < len(pyramid):
                    refined.append(layer(pyramid[i], pyramid[i + 1]))
                else:
                    refined.append(pyramid[i])
            pyramid = refined
        return pyramid


class TopKRouter(nn.Module):
    def __init__(self, in_dim, num_experts, top_k=2):
        super().__init__()
        self.gate = nn.Linear(in_dim, num_experts)
        self.top_k = top_k

    def forward(self, x):
        logits = self.gate(x)
        topk_vals, topk_idx = torch.topk(logits, self.top_k, dim=-1)
        weights = F.softmax(topk_vals, dim=-1)
        return weights, topk_idx


class Expert(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim))

    def forward(self, x):
        return self.net(x)


class ClassificationDecoder(nn.Module):
    """MoE classification decoder that returns logits + wound_embed for WTCS."""

    def __init__(self, in_dim, num_classes, num_experts=5, top_k=2, embed_dim=128):
        super().__init__()
        self.scale_attn = nn.Sequential(
            nn.Linear(in_dim, 4), nn.Softmax(dim=-1))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.router = TopKRouter(in_dim, num_experts, top_k)
        self.experts = nn.ModuleList([
            Expert(in_dim, in_dim * 2, num_classes) for _ in range(num_experts)])
        self.embed_proj = nn.Linear(in_dim, embed_dim)

    def forward(self, pyramid):
        pooled = [self.pool(f).flatten(1) for f in pyramid]
        stacked = torch.stack(pooled, dim=1)
        scale_w = self.scale_attn(stacked.mean(dim=1))
        fused = (stacked * scale_w.unsqueeze(-1)).sum(dim=1)

        weights, idx = self.router(fused)
        expert_outs = torch.stack([e(fused) for e in self.experts], dim=1)
        idx_exp = idx.unsqueeze(-1).expand(-1, -1, expert_outs.shape[-1])
        selected = torch.gather(expert_outs, 1, idx_exp)
        logits = (selected * weights.unsqueeze(-1)).sum(dim=1)

        wound_embed = self.embed_proj(fused)
        return logits, wound_embed


class SegmentationDecoder_v3(nn.Module):
    """Segmentation decoder with FiLM conditioning from classification (WTCS)."""

    def __init__(self, in_channels, cond_dim, num_levels=4):
        super().__init__()
        ch = in_channels
        self.film_layers = nn.ModuleList()
        self.upsample_blocks = nn.ModuleList()

        for i in range(num_levels):
            self.film_layers.append(FiLMConditioner(cond_dim, ch))
            out_ch = ch // 2 if i < num_levels - 1 else ch // 2
            self.upsample_blocks.append(nn.Sequential(
                nn.ConvTranspose2d(ch, out_ch, 4, stride=2, padding=1),
                nn.BatchNorm2d(out_ch), nn.GELU()))
            ch = out_ch

        self.final = nn.Conv2d(ch, 1, 1)

    def forward(self, pyramid, wound_embed, target_size=None):
        x = pyramid[0]
        for i, (film, up) in enumerate(zip(self.film_layers, self.upsample_blocks)):
            B, C, H, W = x.shape
            x_flat = x.flatten(2).transpose(1, 2)
            x_cond = film(x_flat, wound_embed)
            x = x_cond.transpose(1, 2).reshape(B, C, H, W)
            x = up(x)

        mask = self.final(x)
        if target_size is not None:
            mask = F.interpolate(mask, size=target_size, mode='bilinear', align_corners=False)
        return mask


class DetectionDecoder_v3(nn.Module):
    """Anchor-free detection decoder."""

    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.BatchNorm2d(in_channels), nn.GELU(),
            nn.Conv2d(in_channels, in_channels // 2, 3, padding=1),
            nn.BatchNorm2d(in_channels // 2), nn.GELU())
        self.box_head = nn.Conv2d(in_channels // 2, 4, 1)
        self.cls_head = nn.Conv2d(in_channels // 2, 1, 1)

    def forward(self, pyramid):
        feat = pyramid[0]
        x = self.conv(feat)
        boxes = self.box_head(x).sigmoid()
        cls = self.cls_head(x)
        return boxes, cls


class WILLIEModel(nn.Module):
    """
    WILLIE v3 unified model (used for MINI variant).
    Shared DINOv2 backbone → FPN → WA-CSA → {cls, seg, det} decoders.
    """

    def __init__(self, config):
        super().__init__()
        self.config = config
        num_classes = config["num_classes"]
        neck_dim = config["neck_dim"]
        cls_embed = config.get("cls_embed_dim", neck_dim)

        self.backbone = DINOv2MultiScale(
            model_name=config["backbone_name"],
            extract_layers=config["extract_layers"])

        self.use_f2dca = config.get("use_convnext", False)

        self.neck = FeaturePyramidNeck(
            in_dim=self.backbone.embed_dim, neck_dim=neck_dim, num_levels=4)

        self.wa_csa = WA_CSA_Stack(
            dim=neck_dim,
            num_heads=config.get("num_heads", 4),
            num_layers=config.get("wa_csa_layers", 2))

        self.cls_decoder = ClassificationDecoder(
            in_dim=neck_dim, num_classes=num_classes,
            num_experts=config.get("num_experts", num_classes),
            top_k=config.get("moe_top_k", 2), embed_dim=cls_embed)

        self.seg_decoder = SegmentationDecoder_v3(
            in_channels=neck_dim, cond_dim=cls_embed, num_levels=4)

        self.det_decoder = DetectionDecoder_v3(in_channels=neck_dim)

    def forward(self, x):
        input_size = x.shape[-2:]
        vit_features = self.backbone(x)
        pyramid = self.neck(vit_features)
        pyramid = self.wa_csa(pyramid)

        cls_logits, wound_embed = self.cls_decoder(pyramid)
        seg_mask = self.seg_decoder(pyramid, wound_embed, target_size=input_size)
        det_boxes, det_cls = self.det_decoder(pyramid)

        return {
            "cls_logits": cls_logits,
            "seg_mask": seg_mask,
            "det_boxes": det_boxes,
            "det_classes": det_cls,
        }


# ── MINI config ──
MINI_CONFIG = dict(
    backbone_name="dinov2_vits14",
    extract_layers=(2, 5, 8, 11),
    neck_dim=128,
    num_classes=5,
    num_heads=4,
    wa_csa_layers=2,
    num_experts=5,
    moe_top_k=2,
    cls_embed_dim=128,
)

print("  ✅ WILLIEModel (MINI) defined (DINOv2-S + FPN + WA-CSA + MoE + WTCS)")


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION D: TRANSFORMS
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n📐 TRANSFORMS")
print("-" * 80)

# ── MINI: 224×224 ──
mini_transform = A.Compose([
    A.Resize(height=256, width=256),
    A.CenterCrop(height=224, width=224),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

# ── BASE/XL: 378×378 ──
base_transform = A.Compose([
    A.Resize(height=420, width=420),
    A.CenterCrop(height=378, width=378),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

# ── TTA augmentations (for ensemble) ──
def get_tta_transforms(img_size=378):
    """Returns list of (name, transform) for Test-Time Augmentation."""
    resize_h = int(img_size * 1.12)  # ~423 for 378
    return [
        ("original", A.Compose([
            A.Resize(height=resize_h, width=resize_h),
            A.CenterCrop(height=img_size, width=img_size),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2()])),
        ("hflip", A.Compose([
            A.Resize(height=resize_h, width=resize_h),
            A.CenterCrop(height=img_size, width=img_size),
            A.HorizontalFlip(p=1.0),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2()])),
        ("vflip", A.Compose([
            A.Resize(height=resize_h, width=resize_h),
            A.CenterCrop(height=img_size, width=img_size),
            A.VerticalFlip(p=1.0),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2()])),
        ("rot90", A.Compose([
            A.Resize(height=resize_h, width=resize_h),
            A.CenterCrop(height=img_size, width=img_size),
            A.Rotate(limit=(90, 90), p=1.0, border_mode=cv2.BORDER_REFLECT),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2()])),
        ("crop90", A.Compose([
            A.Resize(height=resize_h, width=resize_h),
            A.CenterCrop(height=int(img_size * 0.9), width=int(img_size * 0.9)),
            A.Resize(height=img_size, width=img_size),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2()])),
    ]

print(f"  ✅ MINI transform: 256→224×224")
print(f"  ✅ BASE/XL transform: 420→378×378")
print(f"  ✅ TTA: 5 augmentations defined")


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION E: MODEL LOADING UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n🔧 MODEL LOADING UTILITIES")
print("-" * 80)

BASE_V2_CFG = BaseV2Config()
XL_CFG = XLConfig()


def load_base_fold_models(variant="cell07", device=DEVICE):
    """Load all 5 fold models for BASE v2 ensemble."""
    models = []
    fold_paths = discovered_checkpoints["base"].get(variant, [])

    if not fold_paths:
        print(f"  ❌ No {variant} fold checkpoints found")
        return models

    for fold_info in fold_paths:
        cfg = BaseV2Config()
        model = WILLIEBaseV2(cfg)
        ckpt = torch.load(fold_info["path"], map_location="cpu", weights_only=False)

        # Handle different checkpoint formats
        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        elif isinstance(ckpt, dict) and "state_dict" in ckpt:
            state_dict = ckpt["state_dict"]
        elif isinstance(ckpt, OrderedDict):
            state_dict = ckpt
        else:
            state_dict = ckpt

        model.load_state_dict(state_dict, strict=False)
        model = model.to(device).eval()
        models.append(model)
        print(f"    ✅ Fold {fold_info['fold']} loaded ({fold_info['size_mb']:.1f} MB)")

    print(f"  📊 Loaded {len(models)} BASE fold models ({variant})")
    return models


def load_xl_model(checkpoint_name=None, device=DEVICE):
    """Load XL model from checkpoint."""
    if not discovered_checkpoints["xl"]:
        print("  ⚠️  No XL checkpoints found, skipping")
        return None

    if checkpoint_name:
        ckpt_info = next((c for c in discovered_checkpoints["xl"]
                          if c["name"] == checkpoint_name), None)
    else:
        ckpt_info = discovered_checkpoints["xl"][0]  # Use first available

    if not ckpt_info:
        print(f"  ❌ XL checkpoint '{checkpoint_name}' not found")
        return None

    print(f"  Loading XL: {ckpt_info['name']} ({ckpt_info['size_mb']:.1f} MB)")
    cfg = XLConfig()
    model = WILLIEXL(cfg)

    ckpt = torch.load(ckpt_info["path"], map_location="cpu", weights_only=False)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    elif isinstance(ckpt, OrderedDict):
        state_dict = ckpt
    else:
        state_dict = ckpt

    model.load_state_dict(state_dict, strict=False)
    model = model.to(device).eval()
    print(f"  ✅ XL model loaded")
    return model


def load_mini_model(checkpoint_path=None, device=DEVICE):
    """Load MINI model from checkpoint."""
    if checkpoint_path and os.path.exists(checkpoint_path):
        model = WILLIEModel(MINI_CONFIG)
        ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        elif isinstance(ckpt, OrderedDict):
            state_dict = ckpt
        else:
            state_dict = ckpt
        model.load_state_dict(state_dict, strict=False)
        model = model.to(device).eval()
        print(f"  ✅ MINI model loaded from {checkpoint_path}")
        return model
    else:
        print(f"  ⚠️  MINI checkpoint not found, skipping")
        return None


def load_xl_test_probs():
    """Load pre-computed XL test probabilities for ensemble."""
    if not xl_probs_ckpts:
        return None

    path = xl_probs_ckpts[0]["path"]
    data = torch.load(path, map_location="cpu", weights_only=False)
    print(f"  ✅ XL test probs loaded from {path}")

    # Inspect what's in the file
    if isinstance(data, dict):
        print(f"     Keys: {list(data.keys())}")
        for k, v in data.items():
            if isinstance(v, (torch.Tensor, np.ndarray)):
                print(f"     {k}: shape={getattr(v, 'shape', 'N/A')}")
            elif isinstance(v, (list, tuple)):
                print(f"     {k}: len={len(v)}")
    elif isinstance(data, torch.Tensor):
        print(f"     Tensor shape: {data.shape}")
    elif isinstance(data, np.ndarray):
        print(f"     Array shape: {data.shape}")

    return data


print("  ✅ load_base_fold_models(variant) → list of models")
print("  ✅ load_xl_model(checkpoint_name) → single model")
print("  ✅ load_xl_test_probs() → pre-computed probs (no model needed)")
print("  ✅ load_mini_model(checkpoint_path) → single model")


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION F: INFERENCE HELPERS
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n🏥 INFERENCE HELPERS")
print("-" * 80)


def preprocess_image(image_path, transform):
    """Load and preprocess a single image."""
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    augmented = transform(image=img)
    return augmented["image"].unsqueeze(0), img  # tensor, original


@torch.no_grad()
def classify_base_ensemble(image_tensor, models, device=DEVICE):
    """Run BASE 5-fold ensemble classification."""
    image_tensor = image_tensor.to(device)
    all_probs = []

    for model in models:
        out = model(image_tensor)
        logits = out["cls_logits"]
        probs = F.softmax(logits, dim=-1)
        all_probs.append(probs.cpu().numpy())

    # Average probabilities across folds
    avg_probs = np.mean(all_probs, axis=0)
    pred_class = np.argmax(avg_probs, axis=-1)[0]
    confidence = avg_probs[0, pred_class]

    return {
        "class_idx": int(pred_class),
        "class_name": IDX_TO_CLASS[int(pred_class)],
        "confidence": float(confidence),
        "probs": avg_probs[0],
        "per_fold_probs": np.array(all_probs)[:, 0, :],
    }


@torch.no_grad()
def classify_base_tta(image_path, models, device=DEVICE):
    """Run BASE 5-fold × 5-TTA ensemble classification."""
    tta_transforms = get_tta_transforms(378)
    all_probs = []

    for tta_name, tta_tf in tta_transforms:
        tensor, _ = preprocess_image(image_path, tta_tf)
        tensor = tensor.to(device)

        for model in models:
            out = model(tensor)
            probs = F.softmax(out["cls_logits"], dim=-1)
            all_probs.append(probs.cpu().numpy()[0])

    avg_probs = np.mean(all_probs, axis=0)
    pred_class = np.argmax(avg_probs)
    confidence = avg_probs[pred_class]

    return {
        "class_idx": int(pred_class),
        "class_name": IDX_TO_CLASS[int(pred_class)],
        "confidence": float(confidence),
        "probs": avg_probs,
        "num_views": len(all_probs),
    }


@torch.no_grad()
def segment_xl(image_tensor, model, device=DEVICE):
    """Run XL segmentation."""
    if model is None:
        return None

    image_tensor = image_tensor.to(device)
    out = model(image_tensor, task="segmentation")
    mask = out.get("seg_mask")
    if mask is not None:
        mask = torch.sigmoid(mask).cpu().numpy()[0, 0]
    return mask


def seg_mask_to_bboxes(mask, threshold=0.5, min_area=100):
    """Convert segmentation mask → bounding boxes via connected components."""
    binary = (mask > threshold).astype(np.uint8)

    # Connected components
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary, connectivity=8)

    bboxes = []
    for i in range(1, num_labels):  # skip background
        x, y, w, h, area = stats[i]
        if area >= min_area:
            bboxes.append({
                "bbox": [x, y, x + w, y + h],  # x1, y1, x2, y2
                "area": int(area),
                "centroid": (float(centroids[i][0]), float(centroids[i][1])),
            })

    return bboxes


@torch.no_grad()
def analyze_wound(image_path, base_models=None, xl_model=None,
                  use_tta=True, device=DEVICE):
    """
    Full hospital pipeline: image → {classification, segmentation, detection}.

    Returns dict with all results.
    """
    result = {
        "image_path": str(image_path),
        "timestamp": datetime.now().isoformat(),
        "classification": None,
        "segmentation": None,
        "detection": None,
    }

    # ── Classification (BASE ensemble, optionally with TTA) ──
    if base_models:
        if use_tta:
            cls_result = classify_base_tta(image_path, base_models, device)
        else:
            tensor, original = preprocess_image(image_path, base_transform)
            cls_result = classify_base_ensemble(tensor, base_models, device)
        result["classification"] = cls_result

    # ── Segmentation + Detection (XL) ──
    if xl_model is not None:
        tensor, original = preprocess_image(image_path, base_transform)
        mask = segment_xl(tensor, xl_model, device)

        if mask is not None:
            # Scale mask to original image size
            orig_h, orig_w = original.shape[:2]
            mask_resized = cv2.resize(mask, (orig_w, orig_h),
                                       interpolation=cv2.INTER_LINEAR)
            result["segmentation"] = {
                "mask": mask_resized,
                "mask_raw": mask,
                "wound_area_pct": float((mask_resized > 0.5).sum() / mask_resized.size * 100),
            }

            # Detection from segmentation
            bboxes = seg_mask_to_bboxes(mask_resized)
            if result["classification"]:
                for bb in bboxes:
                    bb["class_name"] = result["classification"]["class_name"]
                    bb["confidence"] = result["classification"]["confidence"]
            result["detection"] = {
                "bboxes": bboxes,
                "num_wounds": len(bboxes),
                "method": "seg→connected_components",
            }

    return result


print("  ✅ preprocess_image(path, transform) → tensor, original")
print("  ✅ classify_base_ensemble(tensor, models) → cls result")
print("  ✅ classify_base_tta(path, models) → cls result with TTA")
print("  ✅ segment_xl(tensor, model) → mask")
print("  ✅ seg_mask_to_bboxes(mask) → bboxes")
print("  ✅ analyze_wound(path, base_models, xl_model) → full result")


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION G: VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n🎨 VISUALIZATION HELPERS")
print("-" * 80)


def visualize_result(image_path, result, save_path=None, show=True):
    """Create clinical visualization panel: original + mask overlay + bboxes."""
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    has_seg = result["segmentation"] is not None
    has_det = result["detection"] is not None and result["detection"]["num_wounds"] > 0
    has_cls = result["classification"] is not None

    n_panels = 1 + int(has_seg) + int(has_det)
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))
    if n_panels == 1:
        axes = [axes]

    # Panel 1: Original + classification
    axes[0].imshow(img)
    axes[0].set_title("Input Image", fontweight="bold")
    if has_cls:
        cls = result["classification"]
        label = f"{cls['class_name']} ({cls['confidence']:.1%})"
        color = "green" if cls["confidence"] > 0.7 else "orange"
        axes[0].text(0.5, -0.08, label, transform=axes[0].transAxes,
                     ha="center", fontsize=14, fontweight="bold",
                     color=color, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
    axes[0].axis("off")

    panel_idx = 1

    # Panel 2: Segmentation overlay
    if has_seg:
        mask = result["segmentation"]["mask"]
        overlay = img.copy()
        wound_mask = (mask > 0.5)
        overlay[wound_mask] = (overlay[wound_mask] * 0.5 +
                               np.array([255, 0, 0]) * 0.5).astype(np.uint8)

        axes[panel_idx].imshow(overlay)
        area_pct = result["segmentation"]["wound_area_pct"]
        axes[panel_idx].set_title(f"Segmentation (area: {area_pct:.1f}%)",
                                   fontweight="bold")
        axes[panel_idx].axis("off")
        panel_idx += 1

    # Panel 3: Detection bboxes
    if has_det:
        axes[panel_idx].imshow(img)
        for bb in result["detection"]["bboxes"]:
            x1, y1, x2, y2 = bb["bbox"]
            rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       linewidth=2, edgecolor="lime",
                                       facecolor="none")
            axes[panel_idx].add_patch(rect)
            label = bb.get("class_name", "wound")
            axes[panel_idx].text(x1, y1 - 5, label, color="lime",
                                  fontsize=10, fontweight="bold",
                                  bbox=dict(boxstyle="round", facecolor="black", alpha=0.5))
        axes[panel_idx].set_title(
            f"Detection ({result['detection']['num_wounds']} wound(s))",
            fontweight="bold")
        axes[panel_idx].axis("off")

    plt.suptitle("WILLIE Hospital Pipeline — Clinical Report",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches="tight", dpi=150)
        print(f"  💾 Saved: {save_path}")
    if show:
        plt.show()
    plt.close()


print("  ✅ visualize_result(path, result) → clinical report panel")


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION H: CELL 1 SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n{'=' * 80}")
print(f"  ✅ CELL 1 COMPLETE — Hospital Pipeline Setup")
print(f"{'=' * 80}")
print(f"""
  MODELS DEFINED:
    • WILLIEModel     (MINI)  — DINOv2-S, 224×224, cls+seg+det
    • WILLIEBaseV2    (BASE)  — DINOv2-B + ConvNeXt-B, 378×378, cls
    • WILLIEXL        (XL)   — ViT-L + ConvNeXt-L + Hiera, 378×378, cls+seg+det

  CHECKPOINTS FOUND:
    • BASE {best_base_variant}: {best_base_count} folds ✅
    • XL model weights: {'✅' if HAS_XL_MODEL else '❌'}
    • XL test probs: {'✅ (can ensemble without loading XL model)' if HAS_XL_PROBS else '❌'}

  LOADERS:
    • load_base_fold_models(variant) → list of 5 models
    • load_xl_model(name) → XL model (if weights exist)
    • load_xl_test_probs() → pre-computed XL probs (for ensemble)
    • load_mini_model(path) → MINI model

  PIPELINE FUNCTION:
    result = analyze_wound(image_path, base_models, xl_model)
    → result["classification"]  (class, confidence, probs)
    → result["segmentation"]    (mask, wound_area)
    → result["detection"]       (bboxes from seg mask)

  NEXT: Cell 2 — Load models & run inference on test set / new images
""")

  06_WILLIE_HospitalPipeline — Cell 1: Setup & Architecture
  Timestamp: 2026-02-16 20:49:50

⚡ Device: cuda
   GPU: Tesla V100-PCIE-32GB (34.1 GB)

📂 Paths:
   Project:     .
   Artifacts:   artifacts/woundshot_v2
   Checkpoints: artifacts/woundshot_v2/checkpoints/transformer
   XL Ckpts:    artifacts/woundshot_v2/checkpoints/xl
   Pipeline:    artifacts/woundshot_v2/hospital_pipeline


🔍 CHECKPOINT DISCOVERY
--------------------------------------------------------------------------------
  ✅ BASE cell07: 5 fold(s) found
       Fold 1: 682.0 MB
       Fold 2: 682.0 MB
       Fold 3: 682.0 MB
       Fold 4: 682.0 MB
       Fold 5: 682.0 MB
  ✅ BASE cell07b: 5 fold(s) found
       Fold 1: 682.0 MB
       Fold 2: 682.0 MB
       Fold 3: 682.0 MB
       Fold 4: 682.0 MB
       Fold 5: 682.0 MB
  ✅ BASE cell07c: 5 fold(s) found
       Fold 1: 682.0 MB
       Fold 2: 682.0 MB
       Fold 3: 682.0 MB
       Fold 4: 682.0 MB
       Fold 5: 682.0 MB

  ✅ XL: 1 file(s) found
       xl_test_prob

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
#  06_WILLIE_HospitalPipeline.ipynb — CELL 2: Load Models & Run Pipeline
# ══════════════════════════════════════════════════════════════════════════════
#
#  Depends on Cell 1 (all architectures, loaders, transforms defined)
#
#  This cell:
#    A. Load test manifest & build test dataset
#    B. Load BASE 5-fold models
#    C. Load XL pre-computed probs
#    D. Run BASE ensemble + TTA on test set
#    E. Run BASE+XL combined ensemble
#    F. Full metrics: accuracy, F1, confusion matrix, ROC, per-class
#    G. Demo pipeline on sample images (clinical visualization)
#    H. Summary
#
# ══════════════════════════════════════════════════════════════════════════════

import gc
from tqdm import tqdm

print("=" * 80)
print("  Cell 2: Load Models & Run Pipeline")
print(f"  Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION A: LOAD TEST MANIFEST
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n📂 SECTION A: LOAD TEST MANIFEST")
print("-" * 80)

# Auto-detect CSV columns
_detect_csv = MANIFESTS_DIR / "cls_train.csv"
assert _detect_csv.exists(), f"Train manifest not found: {_detect_csv}"

_sample_df = pd.read_csv(_detect_csv, nrows=5)
_cols = _sample_df.columns.tolist()
print(f"  Available columns: {_cols}")

# Image column
IMG_COL = next((c for c in _cols if c in [
    "image_path", "img_path", "path", "filepath"]), None)
if IMG_COL is None:
    IMG_COL = [c for c in _cols if "image" in c.lower() or "path" in c.lower()][0]

# Label column (integer)
LABEL_COL = next((c for c in _cols if c in [
    "unified_label", "label", "class_idx"]), None)
if LABEL_COL is None:
    LABEL_COL = [c for c in _cols if "label" in c.lower()][0]

# Class name column (string) — optional
CLASS_COL = next((c for c in _cols if c in [
    "unified_class", "class_name", "wound_type", "class"]), None)

print(f"  📋 IMG_COL={IMG_COL}, LABEL_COL={LABEL_COL}, CLASS_COL={CLASS_COL}")

# Load all manifests
def load_manifest(csv_path):
    """Load manifest CSV → (paths, labels) arrays."""
    df = pd.read_csv(csv_path)

    # Handle string labels → int
    if df[LABEL_COL].dtype == object:
        class_map = {name: i for i, name in enumerate(CLASS_NAMES)}
        class_map["no wound"] = 4
        class_map["BG"] = 4
        df["_label_int"] = df[LABEL_COL].map(class_map)
        df = df.dropna(subset=["_label_int"]).reset_index(drop=True)
        label_col = "_label_int"
    else:
        label_col = LABEL_COL

    paths, labels = [], []
    missing = 0
    for _, row in df.iterrows():
        img_path = str(row[IMG_COL])
        if os.path.exists(img_path):
            paths.append(img_path)
            labels.append(int(row[label_col]))
        else:
            missing += 1
    if missing > 0:
        print(f"    ⚠️  {missing} images not found (skipped)")
    return np.array(paths), np.array(labels, dtype=np.int64)


# Load test set
test_csv = MANIFESTS_DIR / "cls_test.csv"
assert test_csv.exists(), f"Test manifest not found: {test_csv}"
test_paths, test_labels = load_manifest(test_csv)

# Also load train+val for reference
train_paths, train_labels = load_manifest(MANIFESTS_DIR / "cls_train.csv")
val_paths, val_labels = load_manifest(MANIFESTS_DIR / "cls_val.csv")

print(f"\n  📊 Dataset sizes:")
print(f"     Train: {len(train_paths)}")
print(f"     Val:   {len(val_paths)}")
print(f"     Test:  {len(test_paths)} (held-out, used for evaluation)")

# Class distribution in test set
print(f"\n  📊 Test set class distribution:")
for cls_idx, name in enumerate(CLASS_NAMES):
    count = (test_labels == cls_idx).sum()
    print(f"     [{cls_idx}] {name:<12s}: {count:>4d} ({count/len(test_labels)*100:5.1f}%)")


# Simple dataset for test inference
class WoundTestDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = list(paths)
        self.labels = list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        if img is None:
            img = np.zeros((378, 378, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        augmented = self.transform(image=img)
        return augmented["image"], self.labels[idx], self.paths[idx]


test_dataset = WoundTestDataset(test_paths, test_labels, base_transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False,
                         num_workers=4, pin_memory=True)
print(f"\n  ✅ Test DataLoader: {len(test_dataset)} images, batch_size=16")


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION B: LOAD BASE 5-FOLD MODELS
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n🏗️  SECTION B: LOAD BASE 5-FOLD MODELS")
print("-" * 80)

# Use best variant discovered in Cell 1
base_models = load_base_fold_models(variant=best_base_variant)
print(f"\n  📊 Loaded {len(base_models)} BASE fold models ({best_base_variant})")


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION C: LOAD XL PRE-COMPUTED PROBS
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n📦 SECTION C: LOAD XL PRE-COMPUTED PROBS")
print("-" * 80)

xl_probs_data = load_xl_test_probs()

# Parse XL probs — structure may vary
xl_test_probs_array = None
if xl_probs_data is not None:
    if isinstance(xl_probs_data, dict):
        # Try common keys
        for key in ["probs", "test_probs", "fold_test_probs", "avg_probs",
                     "tta_probs", "ensemble_probs", "predictions"]:
            if key in xl_probs_data:
                val = xl_probs_data[key]
                if isinstance(val, torch.Tensor):
                    val = val.numpy()
                if isinstance(val, np.ndarray) and val.ndim == 2 and val.shape[-1] == NUM_CLASSES:
                    xl_test_probs_array = val
                    print(f"  ✅ Found XL probs under key '{key}': shape={val.shape}")
                    break

        # If not found with standard keys, inspect all
        if xl_test_probs_array is None:
            print(f"  🔍 Searching all keys for probability arrays...")
            for key, val in xl_probs_data.items():
                if isinstance(val, torch.Tensor):
                    val = val.numpy()
                if isinstance(val, np.ndarray):
                    print(f"     {key}: shape={val.shape}, dtype={val.dtype}")
                    if val.ndim == 2 and val.shape[-1] == NUM_CLASSES:
                        xl_test_probs_array = val
                        print(f"  ✅ Using key '{key}' as XL probs")
                        break
                elif isinstance(val, (list, tuple)) and len(val) > 0:
                    arr = np.array(val)
                    print(f"     {key}: converted shape={arr.shape}")
                    if arr.ndim == 2 and arr.shape[-1] == NUM_CLASSES:
                        xl_test_probs_array = arr
                        print(f"  ✅ Using key '{key}' as XL probs")
                        break

        # Also check for labels in the data
        xl_labels = None
        for key in ["labels", "test_labels", "true_labels", "targets"]:
            if key in xl_probs_data:
                val = xl_probs_data[key]
                if isinstance(val, torch.Tensor):
                    val = val.numpy()
                xl_labels = np.array(val, dtype=np.int64)
                print(f"  📋 XL labels found under '{key}': {len(xl_labels)} samples")
                break

    elif isinstance(xl_probs_data, torch.Tensor):
        xl_test_probs_array = xl_probs_data.numpy()
        print(f"  ✅ XL probs tensor: shape={xl_test_probs_array.shape}")
    elif isinstance(xl_probs_data, np.ndarray):
        xl_test_probs_array = xl_probs_data
        print(f"  ✅ XL probs array: shape={xl_test_probs_array.shape}")

if xl_test_probs_array is not None:
    print(f"\n  📊 XL probs: {xl_test_probs_array.shape[0]} samples × {xl_test_probs_array.shape[1]} classes")
    xl_preds = xl_test_probs_array.argmax(axis=1)
    # Quick check: does sample count match test set?
    if xl_test_probs_array.shape[0] == len(test_labels):
        xl_acc = accuracy_score(test_labels, xl_preds) * 100
        xl_f1 = f1_score(test_labels, xl_preds, average="macro") * 100
        print(f"  📊 XL standalone: acc={xl_acc:.2f}% f1={xl_f1:.2f}%")
    else:
        print(f"  ⚠️  XL probs has {xl_test_probs_array.shape[0]} samples, "
              f"test set has {len(test_labels)} — may need alignment")
else:
    print(f"  ⚠️  Could not extract XL probability array")


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION D: RUN BASE ENSEMBLE ON TEST SET
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n🔬 SECTION D: BASE ENSEMBLE ON TEST SET")
print("-" * 80)

# D.1: Simple ensemble (no TTA) — fast
print(f"\n  D.1: Simple fold ensemble (no TTA)...")
base_all_probs = []  # [n_folds, n_samples, n_classes]

for fold_idx, model in enumerate(base_models):
    fold_probs = []
    model.eval()
    with torch.no_grad():
        for batch_imgs, batch_labels, batch_paths in test_loader:
            batch_imgs = batch_imgs.to(DEVICE)
            out = model(batch_imgs)
            probs = F.softmax(out["cls_logits"], dim=-1)
            fold_probs.append(probs.cpu().numpy())

    fold_probs = np.concatenate(fold_probs, axis=0)
    base_all_probs.append(fold_probs)
    preds = fold_probs.argmax(axis=1)
    acc = accuracy_score(test_labels, preds) * 100
    print(f"    Fold {fold_idx+1}: acc={acc:.2f}%")

base_all_probs = np.stack(base_all_probs)  # [5, N, 5]
base_ensemble_probs = base_all_probs.mean(axis=0)  # [N, 5]
base_ensemble_preds = base_ensemble_probs.argmax(axis=1)
base_ensemble_acc = accuracy_score(test_labels, base_ensemble_preds) * 100
base_ensemble_f1 = f1_score(test_labels, base_ensemble_preds, average="macro") * 100

print(f"\n  📊 BASE 5-fold ensemble: acc={base_ensemble_acc:.2f}% f1={base_ensemble_f1:.2f}%")

# D.2: TTA ensemble (slower but better)
print(f"\n  D.2: BASE 5-fold × 5-TTA ensemble...")
tta_list = get_tta_transforms(378)
base_tta_all_probs = []  # will accumulate all fold×tta probs

for fold_idx, model in enumerate(base_models):
    model.eval()
    for tta_name, tta_tf in tta_list:
        tta_dataset = WoundTestDataset(test_paths, test_labels, tta_tf)
        tta_loader = DataLoader(tta_dataset, batch_size=16, shuffle=False,
                                num_workers=4, pin_memory=True)
        tta_probs = []
        with torch.no_grad():
            for batch_imgs, _, _ in tta_loader:
                batch_imgs = batch_imgs.to(DEVICE)
                out = model(batch_imgs)
                probs = F.softmax(out["cls_logits"], dim=-1)
                tta_probs.append(probs.cpu().numpy())

        tta_probs = np.concatenate(tta_probs, axis=0)
        base_tta_all_probs.append(tta_probs)

    print(f"    Fold {fold_idx+1} × 5 TTA done")

base_tta_all_probs = np.stack(base_tta_all_probs)  # [25, N, 5]
base_tta_ensemble_probs = base_tta_all_probs.mean(axis=0)  # [N, 5]
base_tta_preds = base_tta_ensemble_probs.argmax(axis=1)
base_tta_acc = accuracy_score(test_labels, base_tta_preds) * 100
base_tta_f1 = f1_score(test_labels, base_tta_preds, average="macro") * 100

print(f"\n  📊 BASE 5-fold × 5-TTA: acc={base_tta_acc:.2f}% f1={base_tta_f1:.2f}%")

# Free GPU memory — models no longer needed for remaining sections
del base_models
gc.collect()
torch.cuda.empty_cache()
print(f"  🧹 GPU memory freed")


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION E: BASE + XL COMBINED ENSEMBLE
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n🤝 SECTION E: BASE + XL COMBINED ENSEMBLE")
print("-" * 80)

results_table = {
    "BASE simple ensemble": {"probs": base_ensemble_probs, "acc": base_ensemble_acc,
                              "f1": base_ensemble_f1},
    "BASE TTA ensemble": {"probs": base_tta_ensemble_probs, "acc": base_tta_acc,
                           "f1": base_tta_f1},
}

if xl_test_probs_array is not None and xl_test_probs_array.shape[0] == len(test_labels):
    # Try multiple weighting schemes for BASE+XL fusion
    weights_to_try = [
        (0.5, 0.5, "equal"),
        (0.6, 0.4, "BASE-heavy"),
        (0.7, 0.3, "BASE-dominant"),
        (0.4, 0.6, "XL-heavy"),
    ]

    best_combined_acc = 0
    best_combined_name = ""

    for w_base, w_xl, name in weights_to_try:
        # Combine with simple ensemble
        combined_simple = w_base * base_ensemble_probs + w_xl * xl_test_probs_array
        preds = combined_simple.argmax(axis=1)
        acc = accuracy_score(test_labels, preds) * 100
        f1 = f1_score(test_labels, preds, average="macro") * 100
        results_table[f"BASE+XL simple ({name})"] = {
            "probs": combined_simple, "acc": acc, "f1": f1}
        print(f"  BASE+XL simple ({name} {w_base}/{w_xl}): acc={acc:.2f}% f1={f1:.2f}%")

        # Combine with TTA ensemble
        combined_tta = w_base * base_tta_ensemble_probs + w_xl * xl_test_probs_array
        preds_tta = combined_tta.argmax(axis=1)
        acc_tta = accuracy_score(test_labels, preds_tta) * 100
        f1_tta = f1_score(test_labels, preds_tta, average="macro") * 100
        results_table[f"BASE-TTA+XL ({name})"] = {
            "probs": combined_tta, "acc": acc_tta, "f1": f1_tta}
        print(f"  BASE-TTA+XL ({name} {w_base}/{w_xl}): acc={acc_tta:.2f}% f1={f1_tta:.2f}%")

        if acc_tta > best_combined_acc:
            best_combined_acc = acc_tta
            best_combined_name = f"BASE-TTA+XL ({name})"

    print(f"\n  🏆 Best combined: {best_combined_name} → {best_combined_acc:.2f}%")
else:
    print(f"  ⚠️  XL probs not available or size mismatch — skipping combined ensemble")
    best_combined_name = "BASE TTA ensemble"
    best_combined_acc = base_tta_acc

# Find overall best
best_name = max(results_table, key=lambda k: results_table[k]["acc"])
best_result = results_table[best_name]
print(f"\n  🏆 OVERALL BEST: {best_name}")
print(f"     Accuracy: {best_result['acc']:.2f}%")
print(f"     F1 Macro: {best_result['f1']:.2f}%")


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION F: FULL METRICS & VISUALIZATIONS
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n📊 SECTION F: FULL METRICS & VISUALIZATIONS")
print("-" * 80)

best_probs = best_result["probs"]
best_preds = best_probs.argmax(axis=1)

# F.1: Classification report
print(f"\n  📋 Classification Report ({best_name}):\n")
print(classification_report(test_labels, best_preds,
      target_names=CLASS_NAMES, digits=4))

# F.2: Per-class metrics
precisions = precision_score(test_labels, best_preds, average=None)
recalls = recall_score(test_labels, best_preds, average=None)
f1s = f1_score(test_labels, best_preds, average=None)

print(f"\n  📊 Per-class breakdown:")
print(f"  {'Class':<12s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s} {'Support':>10s}")
print(f"  {'-'*52}")
for i, name in enumerate(CLASS_NAMES):
    support = (test_labels == i).sum()
    print(f"  {name:<12s} {precisions[i]:>10.4f} {recalls[i]:>10.4f} "
          f"{f1s[i]:>10.4f} {support:>10d}")

# F.3: AUC-ROC
try:
    from sklearn.preprocessing import label_binarize
    test_labels_bin = label_binarize(test_labels, classes=list(range(NUM_CLASSES)))
    auc_per_class = []
    for i in range(NUM_CLASSES):
        auc_i = roc_auc_score(test_labels_bin[:, i], best_probs[:, i])
        auc_per_class.append(auc_i)
    mean_auc = np.mean(auc_per_class)
    print(f"\n  📊 AUC-ROC per class:")
    for i, name in enumerate(CLASS_NAMES):
        print(f"     {name:<12s}: {auc_per_class[i]:.4f}")
    print(f"     {'Mean AUC':<12s}: {mean_auc:.4f}")
except Exception as e:
    mean_auc = 0
    auc_per_class = []
    print(f"  ⚠️  AUC calculation failed: {e}")

# F.4: Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

cm = confusion_matrix(test_labels, best_preds)
im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
axes[0].set_title(f"Confusion Matrix\n{best_name}\nAcc={best_result['acc']:.2f}%",
                   fontweight="bold")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")
axes[0].set_xticks(range(NUM_CLASSES))
axes[0].set_yticks(range(NUM_CLASSES))
axes[0].set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
axes[0].set_yticklabels(CLASS_NAMES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        axes[0].text(j, i, str(cm[i, j]), ha="center", va="center", color=color, fontsize=12)
plt.colorbar(im, ax=axes[0], shrink=0.8)

# F.5: ROC Curves
if auc_per_class:
    colors = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00"]
    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(test_labels_bin[:, i], best_probs[:, i])
        axes[1].plot(fpr, tpr, color=colors[i % len(colors)], lw=2,
                     label=f"{CLASS_NAMES[i]} (AUC={auc_per_class[i]:.3f})")
    axes[1].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title(f"ROC Curves (Mean AUC={mean_auc:.4f})", fontweight="bold")
    axes[1].legend(loc="lower right", fontsize=9)
    axes[1].grid(True, alpha=0.3)

plt.suptitle("WILLIE Hospital Pipeline — Test Set Evaluation",
             fontsize=14, fontweight="bold")
plt.tight_layout()
cm_path = PIPELINE_DIR / "confusion_matrix_roc.png"
plt.savefig(cm_path, bbox_inches="tight", dpi=150)
plt.show()
plt.close()
print(f"  💾 Saved: {cm_path}")

# F.6: Results comparison table
print(f"\n\n  ══════════════════════════════════════════════════════════════")
print(f"  RESULTS COMPARISON — All Configurations Tested")
print(f"  ══════════════════════════════════════════════════════════════")
print(f"\n  {'Configuration':<35s} {'Accuracy':>10s} {'F1 Macro':>10s}")
print(f"  {'-'*55}")
for name, res in sorted(results_table.items(), key=lambda x: -x[1]["acc"]):
    marker = " 🏆" if name == best_name else ""
    print(f"  {name:<35s} {res['acc']:>9.2f}% {res['f1']:>9.2f}%{marker}")


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION G: DEMO PIPELINE ON SAMPLE IMAGES
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n🏥 SECTION G: DEMO PIPELINE ON SAMPLE IMAGES")
print("-" * 80)

# Reload BASE models for demo (we freed them earlier)
print(f"  Reloading BASE models for demo...")
demo_base_models = load_base_fold_models(variant=best_base_variant)

# Pick 1 sample per class from test set
demo_indices = []
for cls_idx in range(NUM_CLASSES):
    cls_mask = test_labels == cls_idx
    cls_indices = np.where(cls_mask)[0]
    if len(cls_indices) > 0:
        demo_indices.append(cls_indices[0])

print(f"  Running pipeline on {len(demo_indices)} sample images (1 per class)...\n")

demo_results = []
for idx in demo_indices:
    img_path = test_paths[idx]
    true_label = CLASS_NAMES[test_labels[idx]]

    # Run pipeline (no XL model loaded, just BASE classification)
    result = analyze_wound(img_path, base_models=demo_base_models,
                          xl_model=None, use_tta=False)

    pred_label = result["classification"]["class_name"]
    confidence = result["classification"]["confidence"]
    correct = "✅" if pred_label == true_label else "❌"

    print(f"  {correct} True: {true_label:<12s} → Pred: {pred_label:<12s} "
          f"(conf={confidence:.1%}) | {os.path.basename(img_path)}")

    # Save visualization
    viz_path = PIPELINE_DIR / f"demo_{true_label}_{os.path.basename(img_path).split('.')[0]}.png"
    visualize_result(img_path, result, save_path=viz_path, show=False)
    demo_results.append(result)

# Free demo models
del demo_base_models
gc.collect()
torch.cuda.empty_cache()


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION H: SAVE RESULTS & SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n{'=' * 80}")
print(f"  ✅ CELL 2 COMPLETE — Hospital Pipeline Evaluation")
print(f"{'=' * 80}")

# Save all results
pipeline_results = {
    "best_config": best_name,
    "best_acc": best_result["acc"],
    "best_f1": best_result["f1"],
    "best_probs": best_probs,
    "best_preds": best_preds,
    "test_labels": test_labels,
    "test_paths": test_paths,
    "base_ensemble_probs": base_ensemble_probs,
    "base_tta_probs": base_tta_ensemble_probs,
    "xl_probs": xl_test_probs_array,
    "mean_auc": mean_auc,
    "auc_per_class": auc_per_class,
    "confusion_matrix": cm,
    "all_results": {k: {"acc": v["acc"], "f1": v["f1"]} for k, v in results_table.items()},
    "class_names": CLASS_NAMES,
    "timestamp": datetime.now().isoformat(),
}
results_path = PIPELINE_DIR / "pipeline_results.pt"
torch.save(pipeline_results, results_path)
print(f"  💾 Saved: {results_path}")

print(f"""
  ══════════════════════════════════════════════════════════════════
  FINAL RESULTS SUMMARY
  ══════════════════════════════════════════════════════════════════

  🏆 Best Config: {best_name}

  📊 CLASSIFICATION (Test Set — {len(test_labels)} samples):
     Accuracy:  {best_result['acc']:.2f}%
     F1 Macro:  {best_result['f1']:.2f}%
     Mean AUC:  {mean_auc:.4f}

  📊 INDIVIDUAL RESULTS:
     BASE simple ensemble:  {results_table['BASE simple ensemble']['acc']:.2f}%
     BASE TTA ensemble:     {results_table['BASE TTA ensemble']['acc']:.2f}%
     XL standalone:         {f'{xl_acc:.2f}%' if xl_test_probs_array is not None and xl_test_probs_array.shape[0] == len(test_labels) else 'N/A'}

  📊 SEGMENTATION: Available via XL model (85.12% Dice)
  📊 DETECTION: seg mask → connected components → bboxes

  💾 OUTPUT FILES:
     {cm_path}
     {results_path}
     {PIPELINE_DIR}/demo_*.png

  ⏭️  NEXT: Cell 3 — Paper visuals & baseline comparison plots
  ══════════════════════════════════════════════════════════════════
""")

  Cell 2: Load Models & Run Pipeline
  Timestamp: 2026-02-16 20:55:35


📂 SECTION A: LOAD TEST MANIFEST
--------------------------------------------------------------------------------
  Available columns: ['image_path', 'original_class', 'unified_label', 'unified_class', 'source', 'azh_split']
  📋 IMG_COL=image_path, LABEL_COL=unified_label, CLASS_COL=unified_class

  📊 Dataset sizes:
     Train: 918
     Val:   162
     Test:  234 (held-out, used for evaluation)

  📊 Test set class distribution:
     [0] diabetic    :   46 ( 19.7%)
     [1] pressure    :   34 ( 14.5%)
     [2] surgical    :   42 ( 17.9%)
     [3] venous      :   62 ( 26.5%)
     [4] no_wound    :   50 ( 21.4%)

  ✅ Test DataLoader: 234 images, batch_size=16


🏗️  SECTION B: LOAD BASE 5-FOLD MODELS
--------------------------------------------------------------------------------
    ✅ Fold 1 loaded (682.0 MB)
    ✅ Fold 2 loaded (682.0 MB)
    ✅ Fold 3 loaded (682.0 MB)
    ✅ Fold 4 loaded (682.0 MB)
    ✅ Fold 5 loade

In [18]:
# ══════════════════════════════════════════════════════════════════════════════
#  06_WILLIE_HospitalPipeline.ipynb — CELL 3: Interactive Upload Platform
# ══════════════════════════════════════════════════════════════════════════════
#
#  Gradio-based web interface for wound analysis.
#  Upload/take a photo → get instant classification + segmentation + detection.
#
#  Features:
#    • Drag & drop / camera upload
#    • BASE 5-fold ensemble classification (91.03% acc)
#    • XL segmentation overlay (85% Dice)
#    • Connected-component wound detection (bounding boxes)
#    • Confidence scores per class
#    • Clinical report panel
#    • Shareable URL for professor / panel demo
#
#  Requires: pip install gradio --break-system-packages
#
# ══════════════════════════════════════════════════════════════════════════════

import subprocess, sys

# Kill any existing Gradio servers
try:
    import gradio as gr
    gr.close_all()
except:
    pass

# Install gradio if not present
try:
    import gradio as gr
    print(f"  ✅ Gradio {gr.__version__} already installed")
except ImportError:
    print("  📦 Installing Gradio...")
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "gradio", "--break-system-packages", "-q"])
    import gradio as gr
    print(f"  ✅ Gradio {gr.__version__} installed")

import gc

print("=" * 80)
print("  Cell 3: WILLIE Hospital Upload Platform")
print(f"  Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION A: LOAD MODELS (persistent — stays in memory while app runs)
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n🏗️  SECTION A: LOADING MODELS")
print("-" * 80)

# Load BASE 5-fold models
platform_base_models = load_base_fold_models(variant=best_base_variant)
print(f"  ✅ {len(platform_base_models)} BASE fold models loaded")

# Load XL probs data (for seg masks if available)
xl_data = load_xl_test_probs()
xl_seg_masks_dict = {}
xl_seg_probs_dict = {}

if xl_data is not None and isinstance(xl_data, dict):
    # Build lookup: image_path → seg mask
    xl_paths = xl_data.get("test_paths", [])
    xl_seg_masks = xl_data.get("xl_seg_masks", [])
    xl_seg_probs = xl_data.get("xl_seg_probs", [])

    if len(xl_paths) > 0 and len(xl_seg_masks) > 0:
        for i, p in enumerate(xl_paths):
            basename = os.path.basename(str(p))
            if i < len(xl_seg_masks):
                mask = xl_seg_masks[i]
                if isinstance(mask, torch.Tensor):
                    mask = mask.numpy()
                if isinstance(mask, np.ndarray):
                    xl_seg_masks_dict[basename] = mask
            if i < len(xl_seg_probs):
                sp = xl_seg_probs[i]
                if isinstance(sp, torch.Tensor):
                    sp = sp.numpy()
                if isinstance(sp, np.ndarray):
                    xl_seg_probs_dict[basename] = sp
        print(f"  ✅ {len(xl_seg_masks_dict)} XL segmentation masks loaded")
    else:
        print(f"  ⚠️  XL seg masks not available in probs file")
else:
    print(f"  ⚠️  XL data not loaded — segmentation will be unavailable")

print(f"\n  🚀 Models ready for inference!")


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION B: ANALYSIS FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

WOUND_COLORS = {
    "diabetic": "#e41a1c",
    "pressure": "#377eb8",
    "surgical": "#4daf4a",
    "venous":   "#984ea3",
    "no_wound": "#999999",
}

WOUND_DESCRIPTIONS = {
    "diabetic": "Diabetic wound — typically found on feet/lower extremities. "
                "Common in patients with diabetes mellitus. Requires careful "
                "monitoring for infection and neuropathy assessment.",
    "pressure": "Pressure ulcer/injury — caused by prolonged pressure on skin. "
                "Common sites: sacrum, heels, elbows. Stage assessment and "
                "pressure redistribution recommended.",
    "surgical": "Surgical wound — post-operative incision or wound site. "
                "Monitor for signs of infection, dehiscence, or delayed healing.",
    "venous":   "Venous ulcer — caused by venous insufficiency, typically on "
                "lower legs near ankles. Compression therapy is standard treatment.",
    "no_wound": "No wound detected — the image does not appear to contain "
                "a wound. Please verify the image quality and try again if needed.",
}


def run_analysis(input_path):
    """
    Main analysis function called by Gradio.

    Args:
        input_path: file path string from Gradio (type="filepath")

    Returns:
        (annotated_image, seg_overlay, classification_text, confidence_dict, clinical_report)
    """
    if input_path is None:
        return None, None, "⚠️ Please upload an image.", {}, "No image provided."

    # Load image from path
    img_np = cv2.imread(str(input_path))
    if img_np is None:
        return None, None, "⚠️ Could not read image.", {}, "Failed to load image."
    img_np = cv2.cvtColor(img_np, cv2.COLOR_BGR2RGB)

    orig_h, orig_w = img_np.shape[:2]

    # Try to match this image to a test set image (for cached XL seg masks)
    input_basename = os.path.basename(str(input_path))
    matched_test_basename = None
    for tp in test_paths:
        tp_base = os.path.basename(str(tp))
        if tp_base == input_basename:
            matched_test_basename = tp_base
            break
    # Also check by resolved path
    if matched_test_basename is None:
        input_resolved = os.path.realpath(str(input_path))
        for tp in test_paths:
            if os.path.realpath(str(tp)) == input_resolved:
                matched_test_basename = os.path.basename(str(tp))
                break

    # ── Classification: BASE 5-fold ensemble ──
    augmented = base_transform(image=img_np)
    tensor = augmented["image"].unsqueeze(0).to(DEVICE)

    all_probs = []
    with torch.no_grad():
        for model in platform_base_models:
            out = model(tensor)
            probs = F.softmax(out["cls_logits"], dim=-1)
            all_probs.append(probs.cpu().numpy()[0])

    avg_probs = np.mean(all_probs, axis=0)
    pred_idx = int(np.argmax(avg_probs))
    pred_class = CLASS_NAMES[pred_idx]
    confidence = float(avg_probs[pred_idx])

    # ── Uncertainty detection (out-of-distribution check) ──
    # If confidence is low AND entropy is high, the image is likely OOD
    entropy = -np.sum(avg_probs * np.log(avg_probs + 1e-8))
    max_entropy = -np.log(1.0 / NUM_CLASSES)  # uniform = max uncertainty
    normalized_entropy = entropy / max_entropy  # 0=certain, 1=clueless

    # Fold agreement: how many folds agree on the prediction?
    fold_preds = [CLASS_NAMES[np.argmax(p)] for p in all_probs]
    fold_agreement = sum(1 for fp in fold_preds if fp == pred_class)

    # Flag as uncertain if:
    #  - confidence < 70% AND entropy > 0.7 (spread across classes)
    #  - OR confidence < 50%
    #  - OR fold agreement <= 2 out of 5
    is_uncertain = (
        (confidence < 0.70 and normalized_entropy > 0.70) or
        confidence < 0.50 or
        fold_agreement <= 2
    )

    if is_uncertain:
        # Override to "uncertain" — likely not a wound or OOD image
        uncertainty_note = (
            f"⚠️ LOW CONFIDENCE — The model is uncertain about this image.\n\n"
            f"Top prediction: {pred_class} ({confidence:.1%}), but "
            f"entropy={normalized_entropy:.2f} (high), "
            f"fold agreement={fold_agreement}/5.\n\n"
            f"This may indicate:\n"
            f"• The image does not contain a wound\n"
            f"• The image is outside the training distribution\n"
            f"• Image quality is too low for reliable analysis\n\n"
            f"**Recommendation**: Please upload a clear, close-up photo "
            f"of the wound area for accurate analysis."
        )

    # Confidence dict for Gradio label component
    confidence_dict = {CLASS_NAMES[i]: float(avg_probs[i]) for i in range(NUM_CLASSES)}

    # ── Segmentation (from cached XL masks if available) ──
    seg_mask = None

    # Try to find cached XL seg mask for this image
    if matched_test_basename and matched_test_basename in xl_seg_masks_dict:
        seg_mask = xl_seg_masks_dict[matched_test_basename]
    elif input_basename in xl_seg_masks_dict:
        seg_mask = xl_seg_masks_dict[input_basename]

    # For Gradio demo: if no match found but we have a global lookup,
    # try fuzzy matching (Gradio renames uploaded files)
    seg_available = seg_mask is not None
    seg_info = ""
    wound_bboxes = []

    if seg_mask is not None:
        # Process mask
        if isinstance(seg_mask, torch.Tensor):
            seg_mask = seg_mask.numpy()

        # Handle different mask formats
        if seg_mask.ndim == 3:
            seg_mask = seg_mask.squeeze()  # Remove batch/channel dim
        if seg_mask.ndim == 3 and seg_mask.shape[0] == 1:
            seg_mask = seg_mask[0]

        # Ensure float [0,1]
        if seg_mask.max() > 1.0:
            seg_mask = seg_mask / 255.0

        # Resize to original image size
        if seg_mask.shape[:2] != (orig_h, orig_w):
            seg_mask = cv2.resize(seg_mask.astype(np.float32), (orig_w, orig_h))

        wound_area_pct = float((seg_mask > 0.5).sum() / seg_mask.size * 100)
        wound_bboxes = seg_mask_to_bboxes(seg_mask, threshold=0.5, min_area=100)
        seg_info = (f"\n  Segmentation: ✅ (XL model, 85.12% Dice)\n"
                    f"  Wound Area:   {wound_area_pct:.1f}% of image\n"
                    f"  Detections:   {len(wound_bboxes)} wound region(s)")
    else:
        seg_info = ("\n  Segmentation: ⚠️ Not available for this image\n"
                    "  (XL seg masks cached for test set images only.\n"
                    "   Load XL model weights for new image analysis.)")

    # ── Build annotated image ──
    annotated = img_np.copy()

    if seg_mask is not None:
        # Overlay mask
        if seg_mask.shape[:2] != (orig_h, orig_w):
            seg_mask_resized = cv2.resize(seg_mask.astype(np.float32),
                                          (orig_w, orig_h))
        else:
            seg_mask_resized = seg_mask

        wound_region = seg_mask_resized > 0.5
        if wound_region.any():
            # Red overlay on wound region
            overlay = annotated.copy()
            overlay[wound_region] = (overlay[wound_region] * 0.5 +
                                     np.array([255, 50, 50]) * 0.5).astype(np.uint8)

            # Contour outline
            contours, _ = cv2.findContours(
                (wound_region * 255).astype(np.uint8),
                cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(overlay, contours, -1, (0, 255, 0), 2)

            # Bounding boxes
            for cnt in contours:
                area = cv2.contourArea(cnt)
                if area > 100:
                    x, y, w, h = cv2.boundingRect(cnt)
                    cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 255, 0), 2)
                    cv2.putText(overlay, f"{pred_class} {confidence:.0%}",
                                (x, max(y - 10, 20)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            annotated = overlay

    # Add classification banner at top
    banner_h = 60
    banner = np.zeros((banner_h, orig_w, 3), dtype=np.uint8)

    if is_uncertain:
        banner[:] = (80, 40, 40)  # dark red = warning
        hex_color = "#ff6600"
        r, g, b = 255, 102, 0
        cv2.rectangle(banner, (0, 0), (8, banner_h), (r, g, b), -1)
        text = f"  UNCERTAIN — {pred_class} {confidence:.0%} (low confidence)"
        cv2.putText(banner, text, (15, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 150, 50), 2)
    else:
        banner[:] = (40, 40, 40)  # dark gray
        hex_color = WOUND_COLORS.get(pred_class, "#999999")
        r, g, b = int(hex_color[1:3], 16), int(hex_color[3:5], 16), int(hex_color[5:7], 16)
        cv2.rectangle(banner, (0, 0), (8, banner_h), (r, g, b), -1)
        text = f"  {pred_class.upper()} — {confidence:.1%} confidence"
        cv2.putText(banner, text, (15, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

    annotated = np.vstack([banner, annotated])

    # ── Clinical report text ──
    report_lines = [
        "═" * 50,
        "  WILLIE — Clinical Analysis Report",
        "═" * 50,
        "",
    ]

    if is_uncertain:
        report_lines += [
            "  ⚠️  WARNING: LOW CONFIDENCE PREDICTION",
            f"  The model is uncertain about this image.",
            f"  Confidence: {confidence:.1%} | Entropy: {normalized_entropy:.2f}",
            f"  Fold agreement: {fold_agreement}/5",
            "",
            "  This likely means the image does not contain",
            "  a recognizable wound, or is outside the training",
            "  distribution. Upload a close-up wound photo.",
            "",
        ]

    report_lines += [
        "  ── CLASSIFICATION ──",
        f"  Prediction:     {pred_class.upper()}",
        f"  Confidence:     {confidence:.1%}",
        f"  Entropy:        {normalized_entropy:.2f} ({'high — uncertain' if normalized_entropy > 0.7 else 'low — confident'})",
        f"  Fold Agreement: {fold_agreement}/5",
        f"  Mean AUC:       0.987 (validated on 234 test images)",
        "",
        "  Probability Distribution:",
    ]
    for i in range(NUM_CLASSES):
        bar_len = int(avg_probs[i] * 30)
        bar = "█" * bar_len + "░" * (30 - bar_len)
        marker = " ◄" if i == pred_idx else ""
        report_lines.append(
            f"    {CLASS_NAMES[i]:<12s} {bar} {avg_probs[i]:.1%}{marker}")

    report_lines += [
        "",
        "  ── SEGMENTATION & DETECTION ──",
        seg_info,
    ]

    if wound_bboxes:
        report_lines.append("")
        report_lines.append("  Detected Wound Regions:")
        for j, bb in enumerate(wound_bboxes):
            x1, y1, x2, y2 = bb["bbox"]
            report_lines.append(
                f"    Region {j+1}: [{x1},{y1}]→[{x2},{y2}] "
                f"(area={bb['area']}px²) — {pred_class}")

    report_lines += [
        "",
        "  ── CLINICAL NOTES ──",
        f"    {WOUND_DESCRIPTIONS[pred_class]}",
        "",
        "  ── MODEL INFORMATION ──",
        f"    Cls Model:    WILLIE-BASE v2 (189M × 5 folds)",
        f"    Seg Model:    WILLIE-XL (622M, 85.12% Dice)",
        f"    Backbone:     DINOv2-ViT-B/14 + ConvNeXt-Base",
        f"    Resolution:   378×378",
        f"    Test Acc:     91.03% (BASE+XL ensemble)",
        "",
        "  Per-fold Confidence:",
    ]
    for i, p in enumerate(all_probs):
        fold_pred = CLASS_NAMES[np.argmax(p)]
        fold_conf = np.max(p)
        agree = "✓" if fold_pred == pred_class else "✗"
        report_lines.append(
            f"    Fold {i+1}: {fold_pred:<12s} {fold_conf:.1%} {agree}")

    report_lines += [
        f"",
        f"  Fold Agreement: {fold_agreement}/5 ({fold_agreement/5:.0%})",
        "",
        f"  ⚠️  This tool is for research purposes only.",
        f"     Always consult a qualified healthcare provider.",
        "═" * 50,
    ]

    clinical_report = "\n".join(report_lines)

    # Classification text
    if is_uncertain:
        cls_text = uncertainty_note
    else:
        cls_text = (f"**{pred_class.upper()}** — {confidence:.1%} confidence\n\n"
                    f"{WOUND_DESCRIPTIONS[pred_class]}")

    # ── Build segmentation overlay image ──
    seg_overlay_img = None
    if seg_mask is not None:
        seg_overlay_img = img_np.copy()
        wound_region = seg_mask > 0.5

        if wound_region.any():
            # Create colored overlay
            color_mask = np.zeros_like(seg_overlay_img)

            # Parse wound color
            hex_color = WOUND_COLORS.get(pred_class, "#e41a1c")
            r = int(hex_color[1:3], 16)
            g = int(hex_color[3:5], 16)
            b = int(hex_color[5:7], 16)

            color_mask[wound_region] = [r, g, b]
            seg_overlay_img = (seg_overlay_img * 0.5 + color_mask * 0.5).astype(np.uint8)

            # Draw contours
            contours, _ = cv2.findContours(
                (wound_region.astype(np.uint8) * 255),
                cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(seg_overlay_img, contours, -1, (0, 255, 0), 2)

            # Draw bboxes
            for j, bb in enumerate(wound_bboxes):
                x1, y1, x2, y2 = bb["bbox"]
                cv2.rectangle(seg_overlay_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                label_text = f"{pred_class} #{j+1}"
                cv2.putText(seg_overlay_img, label_text, (x1, max(y1 - 8, 20)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            # Add area info
            wound_area_pct = float(wound_region.sum() / wound_region.size * 100)
            cv2.putText(seg_overlay_img,
                        f"Wound area: {wound_area_pct:.1f}% | {len(wound_bboxes)} region(s)",
                        (10, orig_h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                        (255, 255, 255), 2)

    return annotated, seg_overlay_img, cls_text, confidence_dict, clinical_report


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION C: BUILD GRADIO INTERFACE
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n🖥️  SECTION C: BUILDING GRADIO INTERFACE")
print("-" * 80)

# Custom CSS for professional medical look
custom_css = """
.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
}
.wound-header {
    text-align: center;
    padding: 20px;
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
    color: white;
    border-radius: 10px;
    margin-bottom: 20px;
}
"""

with gr.Blocks(css=custom_css, title="WILLIE — Hospital Wound Analysis") as demo:

    # ── Header ──
    gr.Markdown("""
    # 🏥 WILLIE — Hospital Wound Analysis Platform

    **Multi-Task Deep Learning for Automated Wound Classification, Segmentation & Detection**

    Upload or take a photo of a wound to receive instant analysis.
    The system uses a 5-fold ensemble of WILLIE-BASE transformers (91.03% test accuracy).

    ---
    """)

    with gr.Row():
        # ── Left: Input ──
        with gr.Column(scale=1):
            gr.Markdown("### 📷 Upload Wound Image")
            input_image = gr.Image(
                label="Drag & drop, click to upload, or use camera",
                type="filepath",
                height=400,
            )
            analyze_btn = gr.Button(
                "🔬 Analyze Wound",
                variant="primary",
                size="lg",
            )
            gr.Markdown("""
            **Supported wound types:**
            - 🔴 Diabetic wounds
            - 🔵 Pressure ulcers
            - 🟢 Surgical wounds
            - 🟣 Venous ulcers
            - ⚪ No wound (healthy skin)
            """)

        # ── Right: Results ──
        with gr.Column(scale=1):
            gr.Markdown("### 📊 Analysis Results")
            output_image = gr.Image(
                label="Annotated Image (Classification + Detection)",
                height=350,
            )
            seg_output_image = gr.Image(
                label="Segmentation Mask Overlay (XL Model)",
                height=350,
            )
            cls_output = gr.Markdown(label="Classification")
            confidence_output = gr.Label(
                label="Class Probabilities",
                num_top_classes=5,
            )

    # ── Bottom: Clinical Report ──
    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📋 Clinical Report")
            report_output = gr.Textbox(
                label="Detailed Analysis",
                lines=25,
                max_lines=35,
                interactive=False,
            )

    # ── Model Info Accordion ──
    with gr.Accordion("ℹ️ Model Architecture & Performance", open=False):
        gr.Markdown(f"""
        ### WILLIE Architecture

        | Component | Details |
        |-----------|---------|
        | **Model** | WILLIE-BASE v2 (5-fold ensemble) |
        | **Parameters** | 189M per model (×5 folds) |
        | **Backbone** | DINOv2-ViT-B/14 + ConvNeXt-Base |
        | **Fusion** | Wound-Aware Frequency-Decomposed Cross-Attention (WA-FDCA) |
        | **Resolution** | 378×378 (27×27 = 729 DINOv2 tokens) |
        | **Test Accuracy** | **91.03%** (BASE+XL ensemble) |
        | **Test F1** | **89.75%** (macro) |
        | **Mean AUC** | **0.987** |
        | **Segmentation** | 85.12% Dice (XL model) |

        ### Novel Contributions
        1. **WA-CSA** — Wound-Aware Cross-Scale Attention (bidirectional, gated)
        2. **WTCS** — Wound-Type Conditioned Segmentation (cls→FiLM→seg)
        3. **MoE** — Mixture-of-Experts classification with wound-type routing
        4. **F²DCA** — Feature-to-Feature Dense Cross-Attention (XL)

        ### Dataset
        - **Training**: {len(train_paths)} images (5-fold cross-validation)
        - **Validation**: {len(val_paths)} images
        - **Test**: {len(test_paths)} images (held-out)
        - **Classes**: 5 (diabetic, pressure, surgical, venous, no_wound)

        ⚠️ **Disclaimer**: This tool is for research and educational purposes only.
        It should not be used as a substitute for professional medical diagnosis.
        """)

    # ── Example images (from test set) ──
    example_paths = []
    for cls_idx in range(NUM_CLASSES):
        cls_mask = test_labels == cls_idx
        cls_indices = np.where(cls_mask)[0]
        if len(cls_indices) > 0:
            example_paths.append(test_paths[cls_indices[0]])

    if example_paths:
        gr.Markdown("### 🖼️ Example Images (click to try)")
        gr.Examples(
            examples=[[p] for p in example_paths],
            inputs=input_image,
            label="Test Set Samples",
        )

    # ── Wire up ──
    analyze_btn.click(
        fn=run_analysis,
        inputs=[input_image],
        outputs=[output_image, seg_output_image, cls_output,
                 confidence_output, report_output],
    )

    # Also trigger on image upload
    input_image.change(
        fn=run_analysis,
        inputs=[input_image],
        outputs=[output_image, seg_output_image, cls_output,
                 confidence_output, report_output],
    )

print("  ✅ Gradio interface built")


# ══════════════════════════════════════════════════════════════════════════════
#  SECTION D: LAUNCH
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n\n{'=' * 80}")
print(f"  🚀 LAUNCHING WILLIE HOSPITAL PLATFORM")
print(f"{'=' * 80}")
print(f"""
  The platform will be available at:
    • Local:  http://localhost:7860
    • Public: A shareable link will be generated (share=True)

  Share the public link with your professor for live demo!

  Press Ctrl+C to stop the server.
""")

# Launch with public sharing enabled
demo.launch(
    share=True,             # Creates public URL
    server_name="0.0.0.0",  # Listen on all interfaces
    server_port=None,        # Auto-find open port
    show_error=True,
    quiet=False,
)

Closing server running on port: 7860
  ✅ Gradio 6.5.1 already installed
  Cell 3: WILLIE Hospital Upload Platform
  Timestamp: 2026-02-16 21:59:14


🏗️  SECTION A: LOADING MODELS
--------------------------------------------------------------------------------
    ✅ Fold 1 loaded (682.0 MB)
    ✅ Fold 2 loaded (682.0 MB)
    ✅ Fold 3 loaded (682.0 MB)
    ✅ Fold 4 loaded (682.0 MB)
    ✅ Fold 5 loaded (682.0 MB)
  📊 Loaded 5 BASE fold models (cell07)
  ✅ 5 BASE fold models loaded
  ✅ XL test probs loaded from artifacts/woundshot_v2/checkpoints/transformer/xl_test_probs.pt
     Keys: ['model_name', 'img_size', 'test_labels', 'test_paths', 'xl_tta_probs', 'xl_raw_probs', 'xl_seg_masks', 'xl_seg_probs', 'xl_tta_acc', 'xl_tta_f1', 'xl_raw_acc', 'num_tta_views']
     test_labels: shape=(234,)
     test_paths: len=234
     xl_tta_probs: shape=(234, 5)
     xl_raw_probs: shape=(234, 5)
     xl_seg_masks: len=234
     xl_seg_probs: len=234
  ✅ 173 XL segmentation masks loaded

  🚀 Models ready 

In [19]:
import gradio as gr
gr.close_all()

Closing server running on port: 7860
